# *__Working on BTCUSD predictions with GRU model.__*

## *__Check first before starting__*

In [4]:
# import os
# os._exit(00)  # Immediately kill the Python process

In [5]:
# Do 'pipenv install ipykernel' if you get error.
print("Kernel is working correctly!")

Kernel is working correctly!


In [1]:
import os

# Change the working directory to the project root
# Working_directory = os.path.normpath("C:/Users/gilda/OneDrive/Documents/_NYCU/MASTER_S_studies/Master\'s Thesis/LABORATORY/_Global_Pytorch/Continual_Learning")
Working_directory = os.path.normpath("C:/Users/james/OneDrive/文件/Continual_Learning")
os.chdir(Working_directory)
print(f"Working directory: {os.getcwd()}")  # Prints the current working directory

Working directory: C:\Users\james\OneDrive\文件\Continual_Learning


## **__All imports__**

In [3]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_interactions import zoom_factory, panhandler
from sklearn.model_selection import train_test_split
import pickle
from ta import trend, momentum, volatility, volume
import math
from scipy.ndimage import gaussian_filter1d
from typing import Callable, Tuple, Union
import shutil
import contextlib
import traceback
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

## __**All functions (For data processing)**__

In [4]:
def ensure_folder(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def plot_with_matplotlib(data: pd.DataFrame, 
                         title: str, 
                         interactive: bool = False, 
                         save_path: str = None, 
                         show_plot: bool = True, 
                         save_matplotlib_object: str = None) -> None:
    """
    Plot data using Matplotlib, with optional interactivity using mpl-interactions.
    
    Parameters:
    - data (pd.DataFrame): The data to plot, must contain 'close' column.
    - title (str): The title of the plot.
    - interactive (bool): If True, enables interactive zoom and pan.
    - save_path (Optional[str]): If provided, saves the plot to this path.
    - show_plot (bool): If True, displays the plot. If False, skips display.
    - save_matplotlib_object (Optional[str]): If provided, saves the Matplotlib figure object to this file path.
    """
    if not all(col in data.columns for col in ['close']):
        raise ValueError("The input DataFrame must contain 'close' column.")

    # Use the default Matplotlib color cycle for the line
    default_blue = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
    print(f"Default blue: {default_blue}")

    # Explicit colors for trends
    trend_colors = {
        0: 'black',
        1: 'yellow',
        2: 'red',
        3: 'green',
        4: default_blue #'purple',
    }
    # unique_trends = [0, -25, -15, 15, 25]
    # colormap = plt.cm.get_cmap('tab10', len(unique_trends))  # Choose 'tab10' or 'Set1' for distinct colors
    # trend_colors = {trend: colormap(i) for i, trend in enumerate(unique_trends)}

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot data as a single connected line, colored by trend
    if 'trend' in data.columns:
        legend_added = set() # Track which trends have already been added to the legend
        prev_idx = data.index[0]
        for idx, row in data.iterrows():
            if idx != prev_idx:
                trend_key = int(row['trend'])  # Convert trend value to int for lookup
                label = f'Trend {trend_key}' if trend_key not in legend_added else None
                ax.plot([prev_idx, idx], 
                        [data.loc[prev_idx, 'close'], row['close']],
                        color=trend_colors[trend_key], 
                        linestyle='-', 
                        # marker='o', 
                        linewidth=1,
                        label=label  # Add label only if it's not in the legend
                )
                legend_added.add(trend_key)  # Mark this trend as added to the legend
            prev_idx = idx

        ax.set_title(f"{title} (Connected, Colored by Trend)")
    else:
        # Default plot if no trend column exists
        ax.plot(data.index, data['close'], label='Closing Price', linestyle='-', marker='o', 
                markersize=2, linewidth=1, color=default_blue, markerfacecolor='green', markeredgecolor='black')
        ax.set_title(title)
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Closing Price (USD)')
    
    # Add a legend manually for trends
    # for trend, color in trend_colors.items():
    #     ax.plot([], [], color=color, label=f'Trend {trend}')
    ax.legend()
    ax.grid()
    
    # Enable interactivity if requested
    if interactive:
        zoom_factory(ax)  # Enable zoom with mouse wheel
        panhandler(fig)   # Enable panning with left-click
        print("Interactive mode enabled. Use mouse wheel to zoom and left click to pan.")

    # Save the plot if a path is provided
    if save_path:
        fig.tight_layout()  # Ensures the layout is clean
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    # Save the Matplotlib figure object
    if save_matplotlib_object:
        with open(save_matplotlib_object, 'wb') as f:
            pickle.dump(fig, f)
        print(f"Matplotlib figure object saved to: {save_matplotlib_object}")

    if show_plot:
        plt.show()
    else:
        print("Plot display skipped.")

def load_and_show_pickle(pickle_file_path: str):
    """
    Load a pickled Matplotlib figure object and display it with optional interactivity.

    Parameters:
    - pickle_file_path (str): Path to the pickled Matplotlib figure file.

    Returns:
    - None
    """
    try:
        # Open the pickle file and load the figure
        with open(pickle_file_path, "rb") as f:
            loaded_fig = pickle.load(f)

        print(f"Figure successfully loaded and displayed from: {pickle_file_path}")

        # Use plt.show() to allow interactivity
        plt.show(block=True)

    except FileNotFoundError:
        print(f"Error: File not found at {pickle_file_path}. Please check the path.")
    except Exception as e:
        print(f"Error loading the pickled figure: {e}")

def save_to_csv(df: pd.DataFrame, file_path: str):
    """
    Save a DataFrame to a CSV file.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data to be saved.
        file_path (str): The file path (including the file name) to save the CSV.

    Returns:
        None
    """
    df.to_csv(file_path)
    # df_to_save = df.copy()
    # df_to_save["date"] = df_to_save.index.strftime('%Y-%m-%d %H:%M:%S')  # Add formatted index as a column
    
    # Save the DataFrame to CSV
    # df_to_save.to_csv(file_path)
    print(f"\nSuccessfully saved data with moving average to CSV: \n\t{file_path}\n")

def read_csv_file(file_path: str, preview_rows: int = 5, 
                  days_towards_end: int = None, 
                  days_from_start: int = None, description: str = ""):
    """
    Reads a CSV file and returns a pandas DataFrame filtered by date range.

    Args:
        file_path (str): The path to the CSV file.
        preview_rows (int): Number of rows to preview (default is 5).
        days_towards_end (int, optional): Number of days from the most recent date to retrieve data.
        days_from_start (int, optional): Number of days from the oldest date of the filtered data to retrieve data.
        description (str): A brief description of the dataset being loaded.
                           Explanation:
                           - To retrieve data from the **end**: Use `days_towards_end`.
                           - To retrieve data from the **start of the filtered range**: Use `days_from_start`.
                           - To retrieve data from the **middle**: Use both:
                             For example, if `days_towards_end=100` and `days_from_start=50`,
                             the function will first filter the last 100 days of the dataset,
                             and then filter the first 50 days from this range.
                             This results in data between the last 100th and the last 50th day.

    Returns:
        DataFrame: The loaded and filtered data from the CSV file.
    """
    try:
        if description:
            print(f"\nDescription: {description}")
        print(f"\nFile path: {file_path}")
        
        # Read the CSV file
        data = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
        
        # Filter by days towards the end
        if days_towards_end is not None:
            last_date = data.index.max()  # Get the most recent date in the dataset
            end_cutoff_date = last_date - pd.Timedelta(days=days_towards_end)
            data = data[data.index >= end_cutoff_date]
            print(f"\nRetrieving data from the past {days_towards_end} days (from {end_cutoff_date.date()} onwards):")
        
        # Filter by days from the start (from the filtered data)
        if days_from_start is not None:
            first_date = data.index.min()  # Get the earliest date in the filtered dataset
            start_cutoff_date = first_date + pd.Timedelta(days=days_from_start)
            data = data[data.index <= start_cutoff_date]
            print(f"\nRetrieving the first {days_from_start} days from the filtered data (up to {start_cutoff_date.date()}):")

        if preview_rows:
            # Print a preview of the data
            print(f"\nPreview of the first {preview_rows} rows:")
            # print(data.head(preview_rows), '\n')
            display(data.head(preview_rows))
            print()

            print(f"\nPreview of the last {preview_rows} rows:")
            # print(data.tail(preview_rows), '\n')
            display(data.tail(preview_rows))
            print()

        return data
    except FileNotFoundError:
        print("Error: File not found. Please check the file path.")
    except pd.errors.EmptyDataError:
        print("Error: The file is empty.")
    except pd.errors.ParserError:
        print("Error: The file could not be parsed. Please check the file format.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def downsample_minute_data(data: pd.DataFrame, n: int) -> pd.DataFrame:
    """
    Downsample minute data into N-minute intervals by retaining every Nth row.

    Parameters:
        data (pd.DataFrame): The original DataFrame with a datetime index.
        n (int): The number of minutes for the downsampling interval.

    Returns:
        pd.DataFrame: Downsampled DataFrame.
    """
    print("\n========---> Downsampling the data! \n")
    data = data.copy()
    # Ensure the index is a DatetimeIndex
    if not isinstance(data.index, pd.DatetimeIndex):
        try:
            data.index = pd.to_datetime(data.index)  # Convert to DatetimeIndex
        except Exception as e:
            raise ValueError("The DataFrame index could not be converted to DatetimeIndex.") from e

    # Filter rows where the minute index modulo N is 0
    downsampled_data = data[data.index.minute % n == 0]

    return downsampled_data

def calculate_log_returns_all_columns(data: pd.DataFrame, exclude_columns: list = [], dropna: bool = True) -> pd.DataFrame:
    """
    Calculate log returns for all numeric columns in a pandas DataFrame,
    excluding specified columns, and removing excluded columns from the returned DataFrame.

    Args:
        data (pd.DataFrame): Input DataFrame containing numeric data.
        exclude_columns (list): List of columns to exclude from log return calculations and the result.
        dropna (bool): Whether to drop rows with NaN values resulting from the calculation.

    Returns:
        pd.DataFrame: DataFrame with log returns for numeric columns,
                      excluding specified columns.
    """
    data = data.copy().drop(columns=exclude_columns)
    columns_to_transform = data.select_dtypes(include=[np.number]).columns
    print(f"columns_to_transform = \n{columns_to_transform}, \nlen(columns_to_transform) = {len(columns_to_transform)}")

    for col in columns_to_transform:
        # Ensure no negative or zero values
        if (data[col] <= 0).any():
            raise ValueError(f"Column '{col}' contains non-positive values. Log returns require strictly positive values.")
        data[col] = np.log(data[col] / data[col].shift(1))

    # Optionally drop rows with NaN values
    return data.dropna() if dropna else data

def created_sequences_2(data: pd.DataFrame, sequence_length: int = 60, sliding_interval: int = 60) -> list:
    """
    Divide the dataset into sequences based on the sequence_length.
    Each sequence must fully cover the window size.

    Args:
    - data (pd.DataFrame): The input DataFrame.
    - sequence_length (int): The window size for sequences.

    Returns:
    - sequences (list): A list of sequences (as DataFrames).
    """
    sequences = []
    for i in range(0, len(data) - sequence_length + 1, sliding_interval):
        # print(f"Processing sequence starting at index: {i}")
        seq = data.iloc[i:i + sequence_length].copy()
        sequences.append(seq)
    # print(f"Total sequences created: {len(sequences)}")
    return sequences

def gaussian_smoothing(data: pd.DataFrame, sigma=2) -> pd.DataFrame:
    """
    Applies Gaussian smoothing to numeric columns in a DataFrame and ensures the index is sorted in ascending order.

    Args:
        data (pd.DataFrame): Input DataFrame.
        sigma (float): Standard deviation for the Gaussian kernel. Defaults to 2.

    Returns:
        pd.DataFrame: A DataFrame with smoothed numeric columns and sorted index.
    """
    # Sort the DataFrame by index in ascending order
    data = data.sort_index(ascending=True).copy()
    for column in data.columns:
        if pd.api.types.is_numeric_dtype(data[column]):  # Only apply to numeric columns
            data[column] = gaussian_filter1d(data[column].values, sigma=sigma)
    return data

def detect_trends_4(
    dataframe: pd.DataFrame, 
    column: str = 'close', 
    lower_threshold: float = 0.001, 
    upper_threshold: float = 0.02,
    reverse_steps: int = 7,
    trends_to_keep: set = {0, 1, 2, 3, 4}  # Default keeps all trends
) -> pd.DataFrame:
    """
    Detects trends based on log return data provided in a specified column and categorizes them into different strength levels.

    This function analyzes time-series data by evaluating cumulative trends in log return values provided in the input DataFrame. 
    It uses three dictionaries (`dic1`, `dic2`, `dic3`) to track different phases of trends, handles multi-step reversals, and 
    classifies trends dynamically based on cumulative product thresholds and specified thresholds for trend strengths.

    Args:
        dataframe (pd.DataFrame): Input DataFrame containing log return data.
        column (str): Column name containing log return values. Defaults to 'close'.
        lower_threshold (float): Threshold for categorizing moderate trends. Defaults to 0.001.
        upper_threshold (float): Threshold for categorizing strong trends. Defaults to 0.02.
        reverse_steps (int): Number of consecutive steps to confirm a trend reversal. Defaults to 7.
        trends_to_keep (set): A set of trend categories to retain; others will be set to 0 (No Trend). Defaults to keeping all trends {0, 1, 2, 3, 4}.

    Returns:
        pd.DataFrame: A DataFrame with an added column:
                    - 'trend': Categorized trend values based on the detected phases:
                        - 0: No trend
                        - 1: Moderate negative trend
                        - 2: Very strong negative trend
                        - 3: Moderate positive trend
                        - 4: Very strong positive trend
                      Any trends not included in `trends_to_keep` will be reset to 0.

    Function Details:
    1. **Input Assumption**:
    - The input DataFrame already contains log return data in the specified column (`column`).

    2. **Trend Tracking**:
    - Uses dictionaries to monitor trends:
        - `dic1`: Tracks the first phase of the trend.
        - `dic2`: Tracks the second phase if a reversal occurs.
        - `dic3`: Tracks the third phase if another reversal occurs.

    3. **Cumulative Product**:
    - Calculates the cumulative product of `(1 + log_return)` from the specified column to evaluate the strength of trends.

    4. **Reversal Handling**:
    - If a trend reversal persists beyond `reverse_steps`, labels are assigned based on the cumulative product tracked in `dic1`.
    - Subsequent reversals are merged or labeled independently if conditions are met.

    5. **Label Assignment**:
    - Labels are dynamically assigned based on cumulative product thresholds for positive and negative trends:
        - Positive trends are categorized as moderate (3) or strong (4).
        - Negative trends are categorized as moderate (1) or strong (2).

    6. **Trend Filtering**:
    - After detecting trends, only those specified in `trends_to_keep` remain unchanged.
    - Any trend category not included in `trends_to_keep` is reset to 0 (No Trend).

    7. **Edge Cases**:
    - Properly handles scenarios where data points are insufficient for trend analysis or when trend phases overlap, ensuring all data points are labeled.
    """
    # Copy to avoid modifying the original DataFrame
    df = dataframe.copy()
    df['trend'] = None  # Default value 
    
    # print("\n#-------------------- Working on 'trend' patterns -----------------------#")
    dic1, dic2, dic3 = None, None, None # Initialize trend tracking dictionaries
    # dic1 = None # {'ids': [], 'last_sign': None, 'cumulative': 1.0}
    
    def assign_label(dictio_, lower_threshold, upper_threshold):
        cumulative = dictio_['cumulative']
        # print(f"cumulative = {cumulative}")
        if cumulative > (1 + upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 4  # Very strong positive
        elif (1 + lower_threshold) < cumulative <= (1 + upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 3  # Moderate positive
        elif (1 - upper_threshold) < cumulative <= (1 - lower_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 1  # Moderate negative
        elif cumulative <= (1 - upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 2  # Very strong negative
        else:
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 0  # No trend
    
    #----------------------- For Loop -----------------------#
    for idx, log_ret in enumerate(df[column]):
        sign = 1 if log_ret > 0 else -1

        if dic1 is None:  # Initialize dic1
            # print(f"\nThis one time condition 'if loop' is running \n")
            dic1 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue
        last_sign = dic1['last_sign']
        if sign == last_sign and dic2 is None:  # Continue same trend
            dic1['ids'].append(idx)
            dic1['last_sign'] = sign
            dic1['cumulative'] *= (1 + log_ret)
            continue

        # 1st Reversal occuring
        if dic2 is None:  # Start dic2
            dic2 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue
        last_sign = dic2['last_sign']
        if sign == last_sign and dic3 is None:  # Continue same trend
            dic2['ids'].append(idx)
            dic2['last_sign'] = sign
            dic2['cumulative'] *= (1 + log_ret)
            if len(dic2['ids']) == reverse_steps:
                assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                # print(f"dic1['cumulative'] = {dic1['cumulative']}, and dic1['ids'] = {dic1['ids']}")
                dic1 = dic2
                dic2 = None
                # print(f"dic1 after trend reversal persisted and dic1 = dic2 = \n{dic1}")
                # print(f"dic2 after being reset: {dic2}\n")
            continue

        # 2nd Reversal occuring
        if dic3 is None:  # Start dic3
            dic3 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue
        last_sign = dic3['last_sign']
        if sign == last_sign: # Continue same trend, there is no dic4 to check if is None
            dic3['ids'].append(idx)
            dic3['last_sign'] = sign
            dic3['cumulative'] *= (1 + log_ret)
            dic_prod = dic2['cumulative'] * dic3['cumulative']
            # if (sign == 1 and dic1['cumulative'] * dic_prod > dic1['cumulative']) or (sign == -1 and dic1['cumulative'] * dic_prod < dic1['cumulative'])):
            if (sign == 1 and dic_prod > 1) or (sign == -1 and dic_prod < 1): # More beautiful
                # Merge dic1, dic2, and dic3
                dic1['ids'] += dic2['ids'] + dic3['ids']
                dic1['last_sign'] = dic3['last_sign']
                dic1['cumulative'] *= dic2['cumulative'] * dic3['cumulative']
                dic2, dic3 = None, None
                continue

            if len(dic3['ids']) == reverse_steps:      
                assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                assign_label(dic2, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                dic1 = dic3
                dic2, dic3 = None, None
                # print(f"dic2 after 2nd trend reversal didn't catch up fast enough, and now \ndic1 = dic3 = {dic1}")
                # print(f"dic3 and dic2 after being reset: {dic3}\n")
            continue
            
        # 3rd Reversal occuring
        assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
        # Reassign values
        dic1 = dic2
        dic2 = dic3
        dic3 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
        # print(f"There was a 3rd trend reversal, and now \ndic1 = dic2 = {dic1}, \ndic2 = dic3 = {dic2}")
        # print(f"dic3 after being reset: {dic3}\n")

    # Assign remaining labels
    if dic1:
        assign_label(dic1, lower_threshold, upper_threshold)
    if dic2:
        assign_label(dic2, lower_threshold, upper_threshold)
    if dic3:
        assign_label(dic3, lower_threshold, upper_threshold)
    # print("\n#-------------------- Returning 'trend' patterns ------------------------#")
    
    # Apply filtering: Keep only selected trends, set others to 0
    df['trend'] = df['trend'].apply(lambda x: x if x in trends_to_keep else 0)

    return df

def split_X_y(sequences: list[pd.DataFrame], 
              target_column: str = 'trend',
              detect_trends_function: Callable[[pd.DataFrame, str, float, float, int, set], pd.DataFrame] = detect_trends_4, 
              column: str = 'close', 
              lower_threshold: float = 0.0009, 
              upper_threshold: float = 0.015,
              reverse_steps: int = 7,
              trends_to_keep: set = {0, 1, 2, 3, 4}) -> Tuple[np.ndarray, np.ndarray]:
    """
    Process sequences to generate features (X) and labels (y) while applying trend detection.

    Args:
    - sequences (list of pd.DataFrame): List of DataFrame sequences.
    - lower_threshold (float): Lower threshold for trend detection.
    - upper_threshold (float): Upper threshold for trend detection.
    - reverse_steps (int): Steps to reverse trends in the sequence.
    - target_column (str): Column name to use as the label (default: 'trend').
    - detect_trends_function (Callable): Function for detecting trends, defaults to `detect_trends_4`.
    - trends_to_keep (set): A set of trend categories to retain; others will be set to 0 (No Trend).

    Returns:
    - X (np.ndarray): Features array of shape (num_sequences, sequence_length, num_features).
    - y (np.ndarray): Labels array of shape (num_sequences,).
    """
    X = []
    y = []
    # count = 0
    for seq in sequences:
        # Apply trend detection on the sequence
        seq = detect_trends_function(seq, column=column, 
                                     lower_threshold=lower_threshold, 
                                     upper_threshold=upper_threshold, 
                                     reverse_steps=reverse_steps,
                                     trends_to_keep=trends_to_keep)
        # if count == 0:
        #     count = 1
        #     print(f"\nseq.head()")
        #     display(seq.head())
        #     print()
        # Extract features (X) and labels (y)
        X.append(seq.drop(columns=[target_column]).values)  # All but the target column
        y.append(seq[target_column].values)  # Target column as labels
        
    # return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)
    return np.array(X), np.array(y)

def process_and_return_splits(
    with_indicators_file_path: str,
    downsampled_data_minutes: int,
    exclude_columns: list[str],
    lower_threshold: float,
    upper_threshold: float,
    reverse_steps: int,
    sequence_length: int,
    sliding_interval: int,
    trends_to_keep: set = {0, 1, 2, 3, 4}  # Default keeps all trends
) -> tuple[
    list[list[float]],  # X_train: List of sequences, each containing a list of features
    list[list[int]],    # y_train: List of sequences, each containing a list of labels
    list[list[float]],  # X_val: List of sequences, each containing a list of features
    list[list[int]],    # y_val: List of sequences, each containing a list of labels
    list[list[float]],  # X_test: List of sequences, each containing a list of features
    list[list[int]]     # y_test: List of sequences, each containing a list of labels
]:
    """
    Processes time-series data from a CSV file and prepares it for machine learning.

    This function performs the following steps:
        1. Reads data from the specified CSV file and sorts it by date in descending order.
        2. Optionally downsamples the data to a lower frequency (e.g., 5-minute intervals).
        3. Applies Gaussian smoothing to reduce noise in the data.
        4. Calculates log returns for all numeric columns, excluding specified columns.
        5. Detects trends based on defined thresholds (`lower_threshold`, `upper_threshold`, and `reverse_steps`).
        6. Filters trends to keep only those specified in `trends_to_keep`, setting others to 0 (No Trend).
        7. Converts the processed data into sequences of a fixed length (`sequence_length`) with a sliding interval.
        8. Splits the sequences into training (80%), validation (10%), and test (10%) sets.
        9. Further splits the sequences into features (`X`) and labels (`y`) for supervised learning.

    Args:
        with_indicators_file_path (str): Path to the CSV file containing the time-series data.
        downsampled_data_minutes (int): Frequency for downsampling the data (e.g., 1 for no downsampling).
        exclude_columns (list[str]): List of column names to exclude from log return calculations.
        lower_threshold (float): Lower threshold for trend detection.
        upper_threshold (float): Upper threshold for trend detection.
        reverse_steps (int): Number of steps for reversing trends in trend detection.
        sequence_length (int): Length of sequences to create from the data.
        sliding_interval (int): Interval for sliding the window when creating sequences.
        trends_to_keep (set): A set of trend categories to retain; others will be set to 0 (No Trend). Defaults to keeping all trends {0, 1, 2, 3, 4}.

    Returns:
        tuple[list[list[float]], list[list[int]], list[list[float]], list[list[int]], list[list[float]], list[list[int]]]:
            A tuple containing:
            - X_train (list[list[float]]): Input sequences for training.
            - y_train (list[list[int]]): Target sequences for training.
            - X_val (list[list[float]]): Input sequences for validation.
            - y_val (list[list[int]]): Target sequences for validation.
            - X_test (list[list[float]]): Input sequences for testing.
            - y_test (list[list[int]]): Target sequences for testing.

    Example:
        X_train, y_train, X_val, y_val, X_test, y_test = process_and_return_splits(
            with_indicators_file_path="data.csv",
            downsampled_data_minutes=5,
            exclude_columns=["volume"],
            lower_threshold=-0.05,
            upper_threshold=0.05,
            reverse_steps=3,
            sequence_length=50,
            sliding_interval=5,
            trends_to_keep={1, 2, 3, 4}  # Only keep categorized trends, set others to 0
        )
    """

    data_retrieved = read_csv_file(with_indicators_file_path, preview_rows=0) # 190 days of data
    data_retrieved = data_retrieved.sort_index(ascending=False)

    #---------------------------------------------------------------------------------------
    if downsampled_data_minutes != 1:
        print("Downsampling the data! \n")
        data_retrieved = downsample_minute_data(data_retrieved, downsampled_data_minutes)
    #---------------------------------------------------------------------------------------

    # Get missing timestamps
    missing_timestamps = pd.date_range(
        start=data_retrieved.index.min(), # Returns smallest/earliest/oldest date
        end=data_retrieved.index.max(),
        freq='1min',  # Use 'min' for a frequency of 1 minute, '30s' for a frequency of 30 seconds
        tz=data_retrieved.index.tz,
    ).difference(data_retrieved.index)
    print(f"\ndata_retrieved - Missing timestamps time: \n{missing_timestamps}") 

    data_gaussian = gaussian_smoothing(data_retrieved, sigma=7)

    # Get missing timestamps
    missing_timestamps = pd.date_range(
        start=data_gaussian.index.min(), # Returns smallest/earliest/oldest date
        end=data_gaussian.index.max(),
        freq='1min',  # Use 'min' for a frequency of 1 minute, '30s' for a frequency of 30 seconds
        tz=data_gaussian.index.tz,
    ).difference(data_gaussian.index)
    print(f"\ndata_gaussian - Missing timestamps time: \n{missing_timestamps}\n")

    data_log_return = calculate_log_returns_all_columns(data_gaussian, exclude_columns=exclude_columns)

    # Get missing timestamps
    missing_timestamps = pd.date_range(
        start=data_log_return.index.min(), # Returns smallest/earliest/oldest date
        end=data_log_return.index.max(),
        freq='1min',  # Use 'min' for a frequency of 1 minute, '30s' for a frequency of 30 seconds
        tz=data_log_return.index.tz,
    ).difference(data_log_return.index)
    print(f"\ndata_log_return - Missing timestamps time: \n{missing_timestamps}\n") 

    # Check if there are missing timestamps
    if missing_timestamps.empty:
        print("No missing timestamps.")
    else:
        for timestamp in missing_timestamps:
            print(f"\nMissing timestamp: {timestamp}")
            
            # Create a placeholder for the missing timestamp
            if timestamp not in data_log_return.index:
                print('Missing')
            
            # Get data before and after the missing timestamp
            before_data = data_log_return[data_log_return.index < timestamp].tail(5)  # 5 data points before
            after_data = data_log_return[data_log_return.index > timestamp].head(5)  # 5 data points after
            
            # Display surrounding data
            if not before_data.empty:
                print("\nData before:")
                print(before_data)
            else:
                print("\nNo data available before the missing timestamp.")
            
            if not after_data.empty:
                print("\nData after:")
                print(after_data)
            else:
                print("\nNo data available after the missing timestamp.")

    sequences = created_sequences_2(data_log_return, sequence_length, sliding_interval)

    # Split sequences into training, validation, and test sets
    train_size = int(len(sequences) * 0.8)
    val_size = int(len(sequences) * 0.1)

    train_sequences = sequences[:train_size]
    val_sequences = sequences[train_size:train_size + val_size]
    test_sequences = sequences[train_size + val_size:]

    print(f"""
    Number of sequences:
        - sequences[0].shape: {sequences[0].shape}
        - Total sequences: {len(sequences)}
        - Train sequences: {len(train_sequences)}
        - Validation sequences: {len(val_sequences)}
        - Test sequences: {len(test_sequences)}
    """)

    # Process train, validation, and test sets
    X_train, y_train = split_X_y(train_sequences, 
                                target_column='trend',
                                detect_trends_function = detect_trends_4,
                                column= 'close',
                                lower_threshold=lower_threshold, 
                                upper_threshold=upper_threshold, 
                                reverse_steps=reverse_steps,
                                trends_to_keep=trends_to_keep)

    X_val, y_val = split_X_y(val_sequences, 
                            target_column='trend',
                            detect_trends_function = detect_trends_4,
                            column= 'close',
                            lower_threshold=lower_threshold, 
                            upper_threshold=upper_threshold, 
                            reverse_steps=reverse_steps,
                            trends_to_keep=trends_to_keep)

    X_test, y_test = split_X_y(test_sequences, 
                            target_column='trend',
                            detect_trends_function = detect_trends_4,
                            column= 'close',
                            lower_threshold=lower_threshold, 
                            upper_threshold=upper_threshold, 
                            reverse_steps=reverse_steps,
                            trends_to_keep=trends_to_keep)

    # Checking X arrays
    for idx, seq in enumerate(X_train):  # Loop through sequences
        for sub_idx, feature_set in enumerate(seq):  # Loop through data points
            for feature_idx, feature in enumerate(feature_set):  # Loop through features
                if not isinstance(feature, (float, np.float32)):  # Check each feature
                    print(f"Unexpected type in X_train at sequence {idx}, data point {sub_idx}, feature {feature_idx}: {type(feature)}")

    # Checking y arrays
    for idx, seq in enumerate(y_train):  # Loop through sequences
        for sub_idx, label in enumerate(seq):  # Loop through data points (labels)
            if not isinstance(label, (int, np.int64)):  # Check each label
                print(f"Unexpected type in y_train at sequence {idx}, data point {sub_idx}: {type(label)}")

    # Checking X arrays
    for idx, seq in enumerate(X_val):  # Loop through sequences
        for sub_idx, feature_set in enumerate(seq):  # Loop through data points
            for feature_idx, feature in enumerate(feature_set):  # Loop through features
                if not isinstance(feature, (float, np.float32)):  # Check each feature
                    print(f"Unexpected type in X_val at sequence {idx}, data point {sub_idx}, feature {feature_idx}: {type(feature)}")
    # Checking y arrays
    for idx, seq in enumerate(y_val):  # Loop through sequences
        for sub_idx, label in enumerate(seq):  # Loop through data points (labels)
            if not isinstance(label, (int, np.int64)):  # Check each label
                print(f"Unexpected type in y_val at sequence {idx}, data point {sub_idx}: {type(label)}")

    # Checking X arrays
    for idx, seq in enumerate(X_test):  # Loop through sequences
        for sub_idx, feature_set in enumerate(seq):  # Loop through data points
            for feature_idx, feature in enumerate(feature_set):  # Loop through features
                if not isinstance(feature, (float, np.float32)):  # Check each feature
                    print(f"Unexpected type in X_test at sequence {idx}, data point {sub_idx}, feature {feature_idx}: {type(feature)}")
    # Checking y arrays
    for idx, seq in enumerate(y_test):  # Loop through sequences
        for sub_idx, label in enumerate(seq):  # Loop through data points (labels)
            if not isinstance(label, (int, np.int64)):  # Check each label
                print(f"Unexpected type in y_test at sequence {idx}, data point {sub_idx}: {type(label)}")

    if isinstance(y_train, np.ndarray) and y_train.dtype == np.object_:
        # Convert to numeric if needed
        y_train = np.array(y_train, dtype=np.int64)

    if isinstance(y_val, np.ndarray) and y_val.dtype == np.object_:
        # Convert to numeric if needed
        y_val = np.array(y_val, dtype=np.int64)

    if isinstance(y_test, np.ndarray) and y_test.dtype == np.object_:
        # Convert to numeric if needed
        y_test = np.array(y_test, dtype=np.int64)

    close_col_index = data_log_return.columns.get_loc('close') # 'date' is set as index so doesnt count as a column
    Number_features = X_train.shape[-1]
    print(f"close_col_index = {close_col_index}")
    print(f"Number_features = {Number_features}")

    return X_train, y_train, X_val, y_val, X_test, y_test, Number_features

def print_class_distribution(y, var_name: str) -> None:
    """
    Prints the class distribution of a label array.

    Args:
        y: Tensor, array, or list of class labels.
        var_name: Name of the variable (for display).
    """
    if isinstance(y, torch.Tensor):
        y = y.cpu().numpy()
    flattened = np.array(y).flatten()

    unique_classes, counts = np.unique(flattened, return_counts=True)
    total = counts.sum()

    header = f"Class Distribution for '{var_name}':"
    line_parts = [
        f"Class {int(c):<3} Percent: {(count / total) * 100:>6.2f}%"
        for c, count in zip(unique_classes, counts)
    ]
    print(header.ljust(40) + " || ".join(line_parts))


## All(Initial) parameters here

In [5]:
ticker = 'BTC-USD'
downsampled_data_minutes = 1

# Step 0 (Again): Identify parameters for trend settings of the loaded data with 1,000 data points
lower_threshold = 0.0009 
upper_threshold = 0.015
reverse_steps = 13


exclude_columns= ['MACD', 'MACD_signal', 'ROC_10', 'OBV', 'AD_Line']
# exclude_columns= []#['open', 'high', 'low', 'MACD', 'MACD_signal', 'BB_middle', 'ROC_10', 'OBV', 'AD_Line']

# Step 3, under ### Correlation Analysis
# Compute correlations with the 'trend' column
# corr = data_trends.corr()
# trend_corr = corr['trend'].sort_values(ascending=False)
strongly_correlated = ['close', 'open', 'SMA_5', 'high', 'low', 'EMA_10', 'SMA_10'] # Strongly correlated (correlation > 0.6)
moderately_correlated = ['BB_middle', 'BB_lower', 'BB_upper', 'RSI_14'] # Moderately correlated (correlation between 0.3 and 0.6)
weakly_correlated = ['SMA_50', 'volume', 'BBW', 'ATR_14'] # Weakly correlated or negligible (correlation <~ 0.3)

exclude_columns += weakly_correlated + moderately_correlated
# print(exclude_columns)


sequence_length = 1000
sliding_interval = 60

## **_Build the GRU Model_**

### GPU details

In [6]:
# If not in Jupyter Notebook
# import subprocess
# result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
# print(result.stdout)

!nvidia-smi

Sun Apr 20 20:29:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.03                 Driver Version: 566.03         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   43C    P8             11W /  200W |    3189MiB /  12282MiB |     16%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Torch and CUDA check

In [7]:
# Check if GPU is available
gpu_available = torch.cuda.is_available()

print("torch.cuda.is_available()", gpu_available)

# If GPU is available, print additional information
if gpu_available:
    print("\nGPU Device Name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("Total CUDA Cores:", torch.cuda.get_device_properties(0).multi_processor_count * 128)  # NVIDIA GPUs often have 128 cores/SM
    print("Current GPU Device:", torch.cuda.current_device())
else:
    print("No GPU detected.")


torch.cuda.is_available() True

GPU Device Name: NVIDIA GeForce RTX 4070
Number of GPUs: 1
Total CUDA Cores: 5888
Current GPU Device: 0


### Model building

#### Bi-Directional GRU with Attention

In [8]:
##################################################
# Base Model (Period 1) – Modified Version for PNN

# But I can load in the weights from the old 
# original version
##################################################
class BiGRUWithAttention(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, output_size: int, num_layers: int, dropout: float = 0.0):
        super(BiGRUWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # Bi-Directional GRU layer
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        # Attention layer
        self.attention_fc = nn.Linear(hidden_size * 2, hidden_size * 2)
        # Fully connected layer for classification (Period 1: classes 0 and 1)
        self.fc = nn.Linear(hidden_size * 2, output_size)
        self.dropout = nn.Dropout(dropout) # Apply dropout (if any) before the fully connected layer
        self.init_weights()

    def init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)  # Xavier initialization for weights
            elif 'bias' in name:
                nn.init.constant_(param, 0)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns intermediate features (after GRU, attention, and dropout)
        to be used as lateral input for the new column.
        """
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size, device=x.device) # Bi-directional: num_layers * 2
        out, _ = self.gru(x, h0)  # (batch, seq_len, hidden_size*2)
        attn_weights = torch.tanh(self.attention_fc(out)) # (batch_size, seq_length, hidden_size * 2)
        features = attn_weights * out  # Element-wise attention application
        features = self.dropout(features) # Apply dropout
        return features  # Shape: (batch, seq_len, hidden_size*2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.forward_features(x)
        out = self.fc(features) # Shape: (batch_size, seq_length, output_size)
        return out


#### New Architecture for PNN

In [9]:
###############################
# PNN New Column (for Period 2)
###############################
class PNNColumn(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, new_output_size: int, num_layers: int, dropout: float = 0.0):
        """
        new_output_size: Number of new classes introduced in this period.
        For Period 2, if you introduce one new class (class 2), new_output_size=1.
        """
        super(PNNColumn, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # New column GRU
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          bidirectional=True, dropout=dropout)
        # Attention layer for new column
        self.attention_fc = nn.Linear(hidden_size * 2, hidden_size * 2)
        # Fully connected layer for new class prediction(s)
        self.fc = nn.Linear(hidden_size * 2, new_output_size)
        self.dropout = nn.Dropout(dropout)
        # Lateral adapter: adapts features from the frozen base column to new column’s space.
        self.lateral_adapter = nn.Linear(hidden_size * 2, hidden_size * 2)
        self.init_weights()

    def init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

    def forward(self, x: torch.Tensor, lateral_features: torch.Tensor) -> torch.Tensor:
        """
        x: Input time-series data.
        lateral_features: Features from the base column (frozen) with shape (batch, seq_len, hidden_size*2).
        """
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)  # (batch, seq_len, hidden_size*2)
        # Adapt and incorporate lateral features
        lateral_adapted = self.lateral_adapter(lateral_features)
        out = out + lateral_adapted
        # Attention mechanism
        attn_weights = torch.tanh(self.attention_fc(out))
        features = attn_weights * out
        features = self.dropout(features)
        new_out = self.fc(features)
        return new_out  # Shape: (batch, seq_len, new_output_size)

####################################
# Progressive Neural Network Wrapper
####################################
class ProgressiveNeuralNetwork(nn.Module):
    def __init__(self, base_model: nn.Module, new_column: PNNColumn):
        """
        Combines a frozen base model (from previous periods) and a new PNN column for the new class.
        The base_model can be either the original BiGRUWithAttention or a previously trained ProgressiveNeuralNetwork.
        The base_model should implement forward_features() to provide its intermediate representation for lateral connections.
        It should also implement get_logits().
        """
        super(ProgressiveNeuralNetwork, self).__init__()
        self.base_model = base_model  # This can be any nn.Module
        self.new_column = new_column  # New column for additional class
        # Freeze the base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns the feature representation from the base model.
        This is required for the lateral adapter in the new column.
        """
        # Check if the base model has forward_features(); if not, error out.
        if not hasattr(self.base_model, 'forward_features'):
            raise NotImplementedError("Base model must implement forward_features() for lateral connections.")
        with torch.no_grad():
            # Will keep calling till it reaches that of BiGRUWithAttention architecture used in period 1
            base_features = self.base_model.forward_features(x) 
        return base_features

    def get_logits(self, x: torch.Tensor, base_features: torch.Tensor) -> torch.Tensor: # x = base_features
        """
        Retrieves the logits from the base model (i.e. for old classes) using its own get_logits() method.
        """
        # print(f"Current base model: {self.base_model.__class__.__name__}")
        if hasattr(self.base_model, 'get_logits'):
            with torch.no_grad():
                base_logits = self.base_model.get_logits(x, base_features)
                new_logits = self.base_model.new_column(x, lateral_features=base_features)
                combined_logits = torch.cat([base_logits, new_logits], dim=-1) # Old logits, Shape: (batch, seq_len, output_size-1)
                return combined_logits
        else:
            with torch.no_grad():
                return self.base_model.fc(base_features)

 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Computes the final combined logits from both the frozen base model and the new column.
        """
        # Get base features, Shape: (batch, seq_len, hidden_size*2)
        base_features = self.forward_features(x) 
        # Retrieve logits for previously learned classes.
        with torch.no_grad():
            base_logits = self.get_logits(x, base_features)  # Shape: (batch, seq_len, num_old_classes)
        # print()

        # # Obtain base model logits using its classification layer (for already learned classes)
        # with torch.no_grad():
        #     base_logits = self.base_model.fc(base_features) # Shape: (batch, seq_len, num_base_classes)

        # New column prediction using the lateral features from the base model, for the new class (e.g. class 2 in period 2)
        new_logits = self.new_column(x, lateral_features=base_features)  # Shape: (batch, seq_len, new_output_size)
        # Combine (concatenate) logits from both columns along the last dimension
        combined_logits = torch.cat([base_logits, new_logits], dim=-1)  # Overall shape: (batch, seq_len, output_size)
        return combined_logits


## Step 6: Train the Models

### Training and validation function

In [ ]:
def print_model_info(model):
    total_params = sum(p.numel() for p in model.parameters())
    param_size_bytes = total_params * 4  # 假設 float32，每個參數佔 4 bytes
    param_size_MB = param_size_bytes / (1024**2)

    print(f"Total Parameters: {total_params}")
    print(f"Model Size (float32): {param_size_MB:.2f} MB")

In [ ]:
def compute_fwt_fixed_verbose(previous_model, init_model, X_val, y_val, known_classes, batch_size=64):
    """
    Corrected FWT computation for token-level labeling with sequence inputs.
    X_val: shape [B, L, F]
    y_val: shape [B, L]
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    previous_model.to(device).eval()
    init_model.to(device).eval()

    B, L, F = X_val.shape
    X_val_flat = X_val  # [B, L, F] (unchanged)
    y_val_flat = y_val.view(-1)  # [B * L]

    # 🔍 找出符合 known_classes 的位置
    mask = torch.isin(y_val_flat, torch.tensor(known_classes, device=y_val.device))
    indices_flat = mask.nonzero(as_tuple=False).squeeze()  # token 級 index

    # 將 token 級 index 映射回 batch 級索引
    batch_indices = indices_flat // L  # 轉為 sequence (樣本) 索引
    batch_indices = batch_indices.unique()

    X_known = X_val[batch_indices]
    y_known = y_val[batch_indices]

    if len(y_known) == 0:
        print(f"⚠️ No validation samples for known classes {known_classes}.")
        return None, None, None

    print(f"📋 Total matching sequences for known classes {known_classes}: {len(y_known)}")

    loader = DataLoader(TensorDataset(X_known, y_known), batch_size=batch_size)

    correct_prev, correct_init, total = 0, 0, 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            if xb.dim() == 2:
                xb = xb.unsqueeze(0)
                yb = yb.unsqueeze(0)

            # [batch, seq, class] → [batch * seq, class]
            out_prev = previous_model(xb).view(-1, previous_model.output_size)
            out_init = init_model(xb).view(-1, init_model.output_size)

            yb_flat = yb.view(-1)
            preds_prev = torch.argmax(out_prev, dim=-1)
            preds_init = torch.argmax(out_init, dim=-1)

            # 只計算 known_classes 的部分
            mask = torch.isin(yb_flat, torch.tensor(known_classes, device=yb.device))
            total_batch = mask.sum().item()

            correct_prev += (preds_prev[mask] == yb_flat[mask]).sum().item()
            correct_init += (preds_init[mask] == yb_flat[mask]).sum().item()
            total += total_batch

    acc_prev = correct_prev / total
    acc_init = correct_init / total
    fwt_value = acc_prev - acc_init

    print(f"\n### 🔍 FWT Debug Info:")
    print(f"- Total evaluated tokens: {total}")
    print(f"- Correct (PrevModel): {correct_prev} / {total} → Acc = {acc_prev:.4f}")
    print(f"- Correct (InitModel): {correct_init} / {total} → Acc = {acc_init:.4f}")
    print(f"- FWT = Acc_prev - Acc_init = {fwt_value:.4f}")

    return fwt_value, acc_prev, acc_init


In [ ]:
def compute_classwise_accuracy(student_logits_flat, y_batch, class_correct, class_total):
    """
    Computes per-class accuracy by accumulating correct and total samples for each class.
    
    student_logits_flat: Model predictions (logits) in shape [batch_size * seq_len, output_size]
    y_batch: True labels in shape [batch_size * seq_len]
    class_correct: Dictionary to store correct predictions per class
    class_total: Dictionary to store total samples per class
    """
    # Convert logits to predicted class indices
    predictions = torch.argmax(student_logits_flat, dim=-1)  # Shape: [batch_size * seq_len]

    # Loop through batch elements to track correct/total per class
    for label, pred in zip(y_batch.cpu().numpy(), predictions.cpu().numpy()):
        if label not in class_total:
            class_total[label] = 0
            class_correct[label] = 0
        class_total[label] += 1
        if label == pred:
            class_correct[label] += 1

###################
# Period 1 Training
###################

# Training and validation function for Period 1.
def train_and_validate(model, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, 
                       use_scheduler=None, num_epochs=10, batch_size=64, 
                       model_saving_folder=None, model_name=None, stop_signal_file=None):
    print("'train_and_validate' function started. \n")
    # Ensure model saving folder exists (deleting existing first if there is one)
    if model_saving_folder and os.path.exists(model_saving_folder):
        # os.rmdir(model_saving_folder) # Only works on empty folders 
        shutil.rmtree(model_saving_folder) # Safely remove all contents
        if not os.path.exists(model_saving_folder):
            print(f"Existing folder has been removed : {model_saving_folder}\n")
    if model_saving_folder and not os.path.exists(model_saving_folder):
        os.makedirs(model_saving_folder)
        
    if not model_saving_folder:
        model_saving_folder = './saved_models'
    if not model_name:
        model_name = 'model'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # Convert data to tensors # Returns a copy, original is safe
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)  # (seqs, seq_len, features)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)    # (seqs, seq_len)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)

    # Create TensorDatasets
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print("y_train:")
    print(type(y_train))
    print(y_train.dtype)
    print(y_train.shape)
    print("X_train:")
    print(type(X_train))
    print(X_train.dtype)
    print(X_train.shape)
    print("\ny_val:")
    print(type(y_val))
    print(y_val.dtype)
    print(y_val.shape)
    print("X_val:")
    print(type(X_val))
    print(X_val.dtype)
    print(X_val.shape)

    # Debug prints for TensorDataset and DataLoader
    print("\nDataset Lengths:")
    print(f"Train Dataset Length: {len(train_dataset)}")
    print(f"Validation Dataset Length: {len(val_dataset)}")

    print("\nDataLoader Batch Sizes:")
    print(f"Number of Batches in Train DataLoader: {len(train_loader)}")
    print(f"Number of Batches in Validation DataLoader: {len(val_loader)}")

    # Additional details for y_train, y_val, and y_test
    print("\ny_train Unique Values and Stats:")
    print(f"Unique values in y_train: {y_train.unique()}")
    print(f"y_train Min: {y_train.min()}, Max: {y_train.max()}")

    print("\ny_val Unique Values and Stats:")
    print(f"Unique values in y_val: {y_val.unique()}")
    print(f"y_val Min: {y_val.min()}, Max: {y_val.max()}")

    # Device check
    print("\nDevice Info:")
    print(f"X_train Device: {X_train.device}")
    print(f"y_train Device: {y_train.device}")
    print(f"X_val Device: {X_val.device}")
    print(f"y_val Device: {y_val.device}\n")

    # Calculate number of batches
    # num_batches = (len(X_train) + batch_size - 1) // batch_size

    global best_results  # Ensure we can modify the external variable if defined outside.
    best_results = []    # Start empty each training run
    model.train()

    for epoch in range(num_epochs):
        epoch_loss = 0
        class_correct = {}  # Dictionary to store correct predictions per class
        class_total = {}  # Dictionary to store total samples per class
        if stop_signal_file and os.path.exists(stop_signal_file):
            print("\nStop signal detected. Exiting training loop safely.\n")
            break
        model.train()
        i=0
        for X_batch, y_batch in train_loader:
            # Reset gradients before forward pass
            optimizer.zero_grad()  # Best practice

            # Forward pass
            outputs = model(X_batch)
            outputs = outputs.view(-1, output_size)
            y_batch = y_batch.view(-1)

            if epoch == 1 and i < 3:
                i += 1
                print(f"\nUnique target values: {y_batch.unique()}")
                print(f"Target dtype: {y_batch.dtype}")
                print(f"Min target: {y_batch.min()}, Max target: {y_batch.max()}")
                print("Unique classes in y_train:", y_train.unique())
                print(f"Unique classes in y_val: {y_val.unique()}\n")
            
            # Compute loss
            loss = criterion(outputs, y_batch)

            # Compute class-wise accuracy (Accumulates values in dict)
            compute_classwise_accuracy(outputs, y_batch, class_correct, class_total)

            # Backward pass and optimization
            # No longer reset gradients here: optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * X_batch.size(0)  # Scale back to total loss
            
        train_loss = epoch_loss / len(train_loader.dataset)

        # Compute per-class training accuracy
        # train_classwise_accuracy = {int(c): (class_correct[c] / class_total[c]) * 100 if class_total[c] > 0 else 0 
        #                            for c in sorted(class_total.keys())}
        train_classwise_accuracy = {int(c): f"{(class_correct[c] / class_total[c]) * 100:.2f}%" if class_total[c] > 0 else "0.00%" 
                                    for c in sorted(class_total.keys())}
        
        # Perform validation at the end of each epoch
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_class_correct = {}
        val_class_total = {}
        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                val_outputs = model(X_val_batch).view(-1, output_size)
                val_labels = y_val_batch.view(-1)
                val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)  # Scale to total loss
                val_predictions = torch.argmax(val_outputs, dim=-1)
                val_correct += (val_predictions == val_labels).sum().item()
                val_total += val_labels.size(0)
                # Compute per-class validation accuracy
                compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
        val_loss /= len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # Compute per-class validation accuracy
        # val_classwise_accuracy = {int(c): (val_class_correct[c] / val_class_total[c]) * 100 if val_class_total[c] > 0 else 0 
        #                          for c in sorted(val_class_total.keys())}
        val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%" 
                                  for c in sorted(val_class_total.keys())}

        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, "
              f"Val Loss: {val_loss:.9f}, "
              f"Val Accuracy: {val_accuracy * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"LR: {optimizer.param_groups[0]['lr']:.9f}")

        # Save current model and update best results if applicable
        current_epoch_info = {
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_classwise_accuracy": train_classwise_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_classwise_accuracy": val_classwise_accuracy,
            'learning_rate': optimizer.param_groups[0]['lr'], # Optimizer state
            "model_path": os.path.join(model_saving_folder, f"{model_name}_epoch_{epoch+1}.pth")
        }

        # Insert this epoch if we have fewer than 5 results
        # or if it beats the lowest of the top 5
        if len(best_results) < 5 or val_accuracy > best_results[-1]["val_accuracy"]:
            if len(best_results) == 5:
                # Remove the worst model from the list, the last (lowest accuracy)
                worst = best_results.pop() 
                if os.path.exists(worst["model_path"]):
                    os.remove(worst["model_path"])
                    print(f"Removed old model with accuracy {worst['val_accuracy']*100:.2f}%, and file was at {worst['model_path']}")
            # Just insert and sort by val_accuracy descending
            best_results.append(current_epoch_info) 
            best_results.sort(key=lambda x: x["val_accuracy"], reverse=True)
            torch.save({ # Save this model
                'epoch': epoch+1,  # Save the current epoch
                'train_loss': train_loss,
                'val_loss': val_loss,
                'model_state_dict': model.state_dict(),  # Model weights
                'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
                'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
            }, current_epoch_info["model_path"])
            print(f"Model saved after epoch {epoch+1} to {current_epoch_info['model_path']} \n")

        if use_scheduler == True:
            # Scheduler step should follow after considering the results (placed after otallher losses)
            scheduler.step(val_loss)


    # Save the final model
    if current_epoch_info:
        final_model_path = os.path.join(model_saving_folder, f"{model_name}_final.pth")
        torch.save({ # Save this model
            'epoch': epoch+1,  # Save the current epoch
            'train_loss': train_loss,
            'val_loss': val_loss,
            'model_state_dict': model.state_dict(),  # Model weights
            'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
            'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
        }, final_model_path)
        print(f"\nFinal model saved to {final_model_path}")

    print("\nTraining complete. \n\nTop 5 Models Sorted by Validation Accuracy: ")
    for res in best_results:        
        print(f"Epoch {res['epoch']}/{num_epochs}, "
              f"Train Loss: {res['train_loss']:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, " 
              f"Val Loss: {res['val_loss']:.9f}, "
              f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"Model Path: {res['model_path']}")
    print('\n')
    
    del X_train, y_train, X_val, y_val, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()

    # Load the checkpoint
    # checkpoint = torch.load("path/to/model_checkpoint.pth")
    # # Restore model state
    # model.load_state_dict(checkpoint['model_state_dict'])
    # # Restore optimizer state
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # # Restore scheduler state (if used)
    # scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    # # Restore epoch and other metadata
    # start_epoch = checkpoint['epoch'] + 1  # Resume from the next epoch
    # loss = checkpoint['loss']  # Optional
    # print(f"Checkpoint loaded. Resuming from epoch {start_epoch}")

###########################################################
# Period 2 and beyond: 
###########################################################

#########################################
# Training Function for Period 2 (PNN)
#########################################
def train_and_validate_pnn(model, output_size, criterion, optimizer, 
                           X_train, y_train, X_val, y_val, scheduler, 
                           use_scheduler=None, num_epochs=10, batch_size=64, 
                           model_saving_folder=None, model_name=None, stop_signal_file=None):
    print("PNN train_and_validate function started.\n")
    # Ensure model saving folder exists (deleting existing first if there is one)
    if model_saving_folder and os.path.exists(model_saving_folder):
        # os.rmdir(model_saving_folder) # Only works on empty folders 
        shutil.rmtree(model_saving_folder) # Safely remove all contents
        if not os.path.exists(model_saving_folder):
            print(f"Existing folder has been removed : {model_saving_folder}\n")
    if model_saving_folder and not os.path.exists(model_saving_folder):
        os.makedirs(model_saving_folder)
        
    if not model_saving_folder:
        model_saving_folder = './saved_models'
    if not model_name:
        model_name = 'model'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # Convert data to tensors # Returns a copy, original is safe
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)  # (seqs, seq_len, features)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)    # (seqs, seq_len)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)

    # Create TensorDatasets
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print("y_train:")
    print(type(y_train))
    print(y_train.dtype)
    print(y_train.shape)
    print("X_train:")
    print(type(X_train))
    print(X_train.dtype)
    print(X_train.shape)
    print("\ny_val:")
    print(type(y_val))
    print(y_val.dtype)
    print(y_val.shape)
    print("X_val:")
    print(type(X_val))
    print(X_val.dtype)
    print(X_val.shape)

    # Debug prints for TensorDataset and DataLoader
    print("\nDataset Lengths:")
    print(f"Train Dataset Length: {len(train_dataset)}")
    print(f"Validation Dataset Length: {len(val_dataset)}")

    print("\nDataLoader Batch Sizes:")
    print(f"Number of Batches in Train DataLoader: {len(train_loader)}")
    print(f"Number of Batches in Validation DataLoader: {len(val_loader)}")

    # Additional details for y_train, y_val, and y_test
    print("\ny_train Unique Values and Stats:")
    print(f"Unique values in y_train: {y_train.unique()}")
    print(f"y_train Min: {y_train.min()}, Max: {y_train.max()}")

    print("\ny_val Unique Values and Stats:")
    print(f"Unique values in y_val: {y_val.unique()}")
    print(f"y_val Min: {y_val.min()}, Max: {y_val.max()}")

    # Device check
    print("\nDevice Info:")
    print(f"X_train Device: {X_train.device}")
    print(f"y_train Device: {y_train.device}")
    print(f"X_val Device: {X_val.device}")
    print(f"y_val Device: {y_val.device}\n")

    # Calculate number of batches
    # num_batches = (len(X_train) + batch_size - 1) // batch_size

    global best_results  # Ensure we can modify the external variable if defined outside.
    best_results = []    # Start empty each training run
    model.train()

    # For validation: use the standard loss (or just compute accuracy)
    criterion_val = nn.CrossEntropyLoss()
    
    print_model_info(model)
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        class_correct = {}  # Dictionary to store correct predictions per class
        class_total = {}  # Dictionary to store total samples per class
        if stop_signal_file and os.path.exists(stop_signal_file):
            print("\nStop signal detected. Exiting training loop safely.\n")
            break
        model.train()
        i=0
        for X_batch, y_batch in train_loader:
            # Reset gradients before forward pass
            optimizer.zero_grad()  # Best practice

            # Forward pass
            outputs = model(X_batch)
            outputs = outputs.view(-1, output_size)
            y_batch = y_batch.view(-1)

            if epoch == 1 and i < 3:
                i += 1
                print(f"\nUnique target values: {y_batch.unique()}")
                print(f"Target dtype: {y_batch.dtype}")
                print(f"Min target: {y_batch.min()}, Max target: {y_batch.max()}")
                print("Unique classes in y_train:", y_train.unique())
                print(f"Unique classes in y_val: {y_val.unique()}\n")
            
            # Compute loss
            loss = criterion(outputs, y_batch)

            # Compute class-wise accuracy (Accumulates values in dict)
            compute_classwise_accuracy(outputs, y_batch, class_correct, class_total)

            # Backward pass and optimization
            # No longer reset gradients here: optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * X_batch.size(0)  # Scale back to total loss
            
        train_loss = epoch_loss / len(train_loader.dataset)

        # Compute per-class training accuracy
        # train_classwise_accuracy = {int(c): (class_correct[c] / class_total[c]) * 100 if class_total[c] > 0 else 0 
        #                            for c in sorted(class_total.keys())}
        train_classwise_accuracy = {int(c): f"{(class_correct[c] / class_total[c]) * 100:.2f}%" 
                                    if class_total[c] > 0 else "0.00%" 
                                    for c in sorted(class_total.keys())}
        
        # Perform validation at the end of each epoch
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_class_correct = {}
        val_class_total = {}
        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                val_outputs = model(X_val_batch).view(-1, output_size)
                val_labels = y_val_batch.view(-1)
                # print("Min/Max of val_outputs:", val_outputs.min().item(), val_outputs.max().item())
                if torch.isnan(val_outputs).any():
                    nan_mask = torch.isnan(val_outputs)
                    nan_indices = torch.nonzero(nan_mask, as_tuple=False)
                    print("Found NaN in val_outputs at indices:")
                    print(nan_indices)
                    print("Corresponding NaN values:")
                    print(val_outputs[nan_mask])
                    print("Corresponding y_val values:")
                    # Note: Since val_outputs is reshaped to (-1, output_size), ensure you match the shape
                    print(y_val_batch.view(-1)[nan_indices[:, 0]])
                loss_val = criterion_val(val_outputs, val_labels)
                # loss_val = criterion(val_outputs, val_labels)
                if torch.isnan(loss_val):
                    print("Loss is nan for this batch")
                    # Print only the parts of val_outputs that are NaN
                    nan_mask = torch.isnan(val_outputs)
                    nan_indices = torch.nonzero(nan_mask, as_tuple=False)
                    print("NaN indices in val_outputs:", nan_indices)
                    print("NaN values in val_outputs:", val_outputs[nan_mask])
                val_loss += loss_val.item() * X_val_batch.size(0)  # Scale to total loss
                val_predictions = torch.argmax(val_outputs, dim=-1)
                val_correct += (val_predictions == val_labels).sum().item()
                val_total += val_labels.size(0)
                # Compute per-class validation accuracy
                compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
        val_loss /= len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # Compute per-class validation accuracy
        # val_classwise_accuracy = {int(c): (val_class_correct[c] / val_class_total[c]) * 100 if val_class_total[c] > 0 else 0 
        #                          for c in sorted(val_class_total.keys())}
        val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" 
                                  if val_class_total[c] > 0 else "0.00%" 
                                  for c in sorted(val_class_total.keys())}

        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, "
              f"Val Loss: {val_loss:.9f}, "
              f"Val Accuracy: {val_accuracy * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"LR: {optimizer.param_groups[0]['lr']:.9f}")

        # Save current model and update best results if applicable
        current_epoch_info = {
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_classwise_accuracy": train_classwise_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_classwise_accuracy": val_classwise_accuracy,
            'learning_rate': optimizer.param_groups[0]['lr'], # Optimizer state
            "model_path": os.path.join(model_saving_folder, f"{model_name}_epoch_{epoch+1}.pth")
        }

        # Insert this epoch if we have fewer than 5 results
        # or if it beats the lowest of the top 5
        if len(best_results) < 5 or val_accuracy > best_results[-1]["val_accuracy"]:
            if len(best_results) == 5:
                # Remove the worst model from the list, the last (lowest accuracy)
                worst = best_results.pop() 
                if os.path.exists(worst["model_path"]):
                    os.remove(worst["model_path"])
                    print(f"Removed old model with accuracy {worst['val_accuracy']*100:.2f}%, and file was at {worst['model_path']}")
            # Just insert and sort by val_accuracy descending
            best_results.append(current_epoch_info) 
            best_results.sort(key=lambda x: x["val_accuracy"], reverse=True)
            torch.save({ # Save this model
                'epoch': epoch+1,  # Save the current epoch
                'train_loss': train_loss,
                'val_loss': val_loss,
                'model_state_dict': model.state_dict(),  # Model weights
                'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
                'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
            }, current_epoch_info["model_path"])
            print(f"Model saved after epoch {epoch+1} to {current_epoch_info['model_path']} \n")

        if use_scheduler == True:
            # Scheduler step should follow after considering the results (placed after otallher losses)
            scheduler.step(val_loss)


    # Save the final model
    if current_epoch_info:
        final_model_path = os.path.join(model_saving_folder, f"{model_name}_final.pth")
        torch.save({ # Save this model
            'epoch': epoch+1,  # Save the current epoch
            'train_loss': train_loss,
            'val_loss': val_loss,
            'model_state_dict': model.state_dict(),  # Model weights
            'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
            'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
        }, final_model_path)
        print(f"\nFinal model saved to {final_model_path}")

    print("\nTraining complete. \n\nTop 5 Models Sorted by Validation Accuracy: ")
    for res in best_results:        
        print(f"Epoch {res['epoch']}/{num_epochs}, "
              f"Train Loss: {res['train_loss']:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, " 
              f"Val Loss: {res['val_loss']:.9f}, "
              f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"Model Path: {res['model_path']}")
    print('\n')
    
    del X_train, y_train, X_val, y_val, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()


### Set parameters and initialize model

In [11]:
print("PyTorch compiled CUDA version:", torch.version.cuda)

PyTorch compiled CUDA version: 12.4


In [12]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Num GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

torch.manual_seed(42)
print("Seeding successful!")


PyTorch Version: 2.4.1+cu124
CUDA Available: True
Num GPUs: 1
GPU Name: NVIDIA GeForce RTX 4070
Seeding successful!


### list_period_files_full_path

In [13]:
pair = 'BTCUSD'
file_name = f'Polygon_{pair}_4Y_1min'  # File name for saving data
# BASE_FOLDER_PATH = f"{Working_directory}/Data/{file_name}"
BASE_FOLDER_PATH = f"Data/{file_name}"
# folder_path=f'{BASE_FOLDER_PATH}/polygon_io/12_USD_Crypto_Pairs/{file_name}'
folder_path=f'{BASE_FOLDER_PATH}'
if not os.path.isdir(folder_path):
    raise FileNotFoundError(f"Directory '{folder_path}' does not exist.")
file_path=f'{folder_path}/{file_name}.csv'
number_days = 190
with_indicators_file_path = os.path.normpath(f'{folder_path}/_{file_name}_{number_days}_days_with_indicators.csv')

# LABORATORY\_Global_Pytorch\Continual_Learning\Data\Polygon_BTCUSD_4Y_1min

list_period_files_full_path = [
    # Period 1
    with_indicators_file_path,
    # Period 2
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20__with_indicators.csv"),
    # Period 3
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26__with_indicators.csv"),
    # Period 4
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04__with_indicators.csv"),
    # Period 5
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11__with_indicators.csv")
]

with_indicators_file_path

'Data\\Polygon_BTCUSD_4Y_1min\\_Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv'

In [14]:
for file in os.listdir(folder_path):
    print(f"Found file: {file}")

# Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv

Found file: Polygon_BTCUSD_4Y_1min.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_525_days.png
Found file: _Polygon_BTCUSD_4Y_1min_190_days.png
Found file: _Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv


In [23]:
X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
    with_indicators_file_path = list_period_files_full_path[0],
    downsampled_data_minutes = downsampled_data_minutes,
    exclude_columns = exclude_columns,
    lower_threshold = lower_threshold,
    upper_threshold = upper_threshold,
    reverse_steps = reverse_steps,
    sequence_length = sequence_length,
    sliding_interval = sliding_interval,
    trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
)

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")

del X_train, y_train, X_val, y_val, X_test, y_test, Number_features
del unique_classes, num_classes



File path: Data\Polygon_BTCUSD_4Y_1min\_Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv

data_retrieved - Missing timestamps time: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')

data_gaussian - Missing timestamps time: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')

columns_to_transform = 
Index(['open', 'high', 'low', 'close', 'SMA_5', 'SMA_10', 'EMA_10'], dtype='object'), 
len(columns_to_transform) = 7

data_log_return - Missing timestamps time: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')

No missing timestamps.

    Number of sequences:
        - sequences[0].shape: (1000, 7)
        - Total sequences: 4543
        - Train sequences: 3634
        - Validation sequences: 454
        - Test sequences: 455
    
close_col_index = 3
Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


### __*All periods data*__

In [24]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""

# Initialize empty lists to combine validation elements
# X_train_all = []
# y_train_all = []
# X_val_all = []
# y_val_all = []
# X_test_all = []
# y_test_all = []

# for path_ in list_period_files_full_path:
#     with contextlib.redirect_stdout(open(os.devnull, 'w')):
#         X_train_, y_train_, X_val_, y_val_, X_test_, y_test_, Number_features = process_and_return_splits(
#             with_indicators_file_path = path_,
#             downsampled_data_minutes = downsampled_data_minutes,
#             exclude_columns = exclude_columns,
#             lower_threshold = lower_threshold,
#             upper_threshold = upper_threshold,
#             reverse_steps = reverse_steps,
#             sequence_length = sequence_length,
#             sliding_interval = sliding_interval,
#             trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
#         )
#     # Combine validation elements
#     X_train_all.extend(X_train_)
#     y_train_all.extend(y_train_)
#     X_val_all.extend(X_val_)
#     y_val_all.extend(y_val_)
#     X_test_all.extend(X_test_)
#     y_test_all.extend(y_test_)
#     # Delete unused variables to save memory
#     del X_train_, y_train_, X_val_, y_val_, X_test_, y_test_

# X_train_all = np.array(X_train_all)
# y_train_all = np.array(y_train_all)
# X_val_all = np.array(X_val_all)
# y_val_all = np.array(y_val_all)
# X_test_all = np.array(X_test_all)
# y_test_all = np.array(y_test_all)


# with contextlib.redirect_stdout(open(os.devnull, 'w')):
#     X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
#         with_indicators_file_path = list_period_files_full_path[0],
#         downsampled_data_minutes = downsampled_data_minutes,
#         exclude_columns = exclude_columns,
#         lower_threshold = lower_threshold,
#         upper_threshold = upper_threshold,
#         reverse_steps = reverse_steps,
#         sequence_length = sequence_length,
#         sliding_interval = sliding_interval,
#         trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
#         # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
#     )
    
# print(f"\nNumber_features = {Number_features}")

# unique_classes = np.unique(y_val)
# num_classes = len(unique_classes)
# print(f"unique_classes = {unique_classes}")
# print(f"num_classes = {num_classes}")


"\n- 'trend': Categorized trend values based on the detected phases:\n    - 0: No trend\n    - 1: Moderate negative trend\n    - 2: Very strong negative trend\n    - 3: Moderate positive trend\n    - 4: Very strong positive trend\n"

### Period 1 --> Training and saving in __*'1st_try'*__ (BiGRUWithAttention, num_layers = 4) ---> Val acc = 98.35 %
### Val-Class-Acc: {0: '98.63%', 1: '97.92%'} <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

In [58]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[0], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1]
num_classes = 2


In [61]:
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------

# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try"))
ensure_folder(model_saving_folder)

# Instantiate the model
class_gru_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers).to(device)

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(class_gru_model.parameters(), lr=0.0001) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

# train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train_all, y_train_all, X_val_all, y_val_all, scheduler, 
train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train, y_train, X_val, y_val, scheduler, 
                   False, num_epochs, batch_size, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model: \n{class_gru_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoader: 57
Number of Batches in Validation DataLoader: 8

y_train Unique Values and Stats:
Unique values in y_train: tensor([0, 1], device='cuda:0')
y_train Min: 0, Max: 1

y_val Unique Values and Stats:
Unique values in y_val: tensor([0, 1], device='cuda:0')
y_val Min: 0, Max: 1

Device Info:
X_train Device: cuda:0
y_train Device: cuda:0
X_val Device: cuda:0
y_val Device: cuda:0

Epoch 1/2000, Train Loss: 0.681195149, 

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def compute_classwise_accuracy(student_logits_flat, y_batch, class_correct, class_total):
    predictions = torch.argmax(student_logits_flat, dim=-1)
    correct_mask = (predictions == y_batch)
    unique_labels = torch.unique(y_batch)

    for label in unique_labels:
        label = label.item()
        if label not in class_total:
            class_total[label] = 0
            class_correct[label] = 0
        label_mask = (y_batch == label)
        class_total[label] += label_mask.sum().item()
        class_correct[label] += (label_mask & correct_mask).sum().item()

def add_val_class_acc(model_copy, output_size, criterion, X_val, y_val, file_path, model_checkpoint, batch_size=64):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_copy.to(device)
    model_copy.eval()

    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)
    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_class_correct = {}
    val_class_total = {}

    with torch.no_grad():
        for X_val_batch, y_val_batch in val_loader:
            val_outputs = model_copy(X_val_batch).view(-1, output_size)
            val_labels = y_val_batch.view(-1)
            val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)
            val_predictions = torch.argmax(val_outputs, dim=-1)
            val_correct += (val_predictions == val_labels).sum().item()
            val_total += val_labels.size(0)
            compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)

    val_accuracy = val_correct / val_total
    val_classwise_accuracy = {
        int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%"
        for c in sorted(val_class_total.keys())
    }

    model_checkpoint.update({
        "val_accuracy": val_accuracy,
        "Val-Class-Acc": val_classwise_accuracy
    })
    torch.save(model_checkpoint, file_path)
    print(f"✅ Updated: {os.path.basename(file_path)} | Acc: {val_accuracy:.4f} | Saved to: {file_path}")

def show_val_accuracy_from_checkpoints(model_folder):
    import torch
    import glob
    import os

    ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

    print(f"\n📝 Summary of val_accuracy and Val-Class-Acc:")
    for path in ckpt_paths:
        checkpoint = torch.load(path, map_location="cpu")
        filename = os.path.basename(path)

        val_acc = checkpoint.get("val_accuracy", None)
        val_class_acc = checkpoint.get("Val-Class-Acc", None)

        if val_acc is not None and val_class_acc is not None:
            print(f"{filename}: val_accuracy = {val_acc:.4f}, Val-Class-Acc: {val_class_acc}")
        else:
            print(f"{filename}: ❌ val_accuracy or Val-Class-Acc not found")


In [ ]:
import os
import glob
import copy

# Reconstruct model with same config
model_template = BiGRUWithAttention(
    input_size=Number_features,
    hidden_size=64,
    output_size=num_classes,
    num_layers=4
)

# Prepare validation data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# Your model folder
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# Loop through and update each model
for file_path in ckpt_paths:
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=num_classes,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

show_val_accuracy_from_checkpoints(
    "Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try"
)


C:\Users\james\AppData\Local\Temp\ipykernel_149096\3997419134.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:

✅ Updated: BiGRUWithAttention_epoch_1550.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1550.pth
✅ Updated: BiGRUWithAttention_epoch_1986.pth | Acc: 0.9835 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1986.pth
✅ Updated: BiGRUWithAttention_epoch_1987.pth | Acc: 0.9835 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1987.pth
✅ Updated: BiGRUWithAttention_epoch_1989.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1989.pth
✅ Updated: BiGRUWithAttention_epoch_1999.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1999.pth
✅ Updated: BiGRUWithAttention_final.pth 

C:\Users\james\AppData\Local\Temp\ipykernel_149096\2472753690.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 1 --> Training and saving in __*'2nd_try'*__ (BiGRUWithAttention, num_layers = 3) ---> Val acc = 98.31 %

In [ ]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[0], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 3  # Number of GRU layers
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/2nd_try"))
ensure_folder(model_saving_folder)

# Instantiate the model
class_gru_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers).to(device)

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(class_gru_model.parameters(), lr=0.0001) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

# train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train_all, y_train_all, X_val_all, y_val_all, scheduler, 
train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train, y_train, X_val, y_val, scheduler, 
                   False, num_epochs, batch_size, model_saving_folder, model_name, stop_signal_file)

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model: \n{class_gru_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoader: 57
Number of Batches in Validation DataLoader: 8

y_train Unique Values and Stats:
Unique values in y_train: tensor([0, 1], device='cuda:0')
y_train Min: 0, Max: 1

y_val Unique Values and Stats:
Unique values in y_val: tensor([0, 1], device='cuda:0')
y_val Min: 0, Max: 1

Device Info:
X_train Device: cuda:0
y_train Device: cuda:0
X_val Device: cuda:0
y_val Device: cuda:0

Epoch 1/2000, Train Loss: 0.010724814, Val Loss: 0.011819

### Period 2 --> Training and saving in __*'1st_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001) ---> Val acc = 96.18 %
### Val-Class-Acc: {0: '99.33%', 1: '94.49%', 2: '86.59%'} <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

In [54]:
###################################################
# Rerun previous cell before running the next cell.
###################################################

# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001
num_epochs= 500 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttentionLoRA' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers)#.to(device)
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")

# Convert data to tensors # Returns a copy, original is safe
X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val, dtype=torch.long).to(device)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

previous_model.to(device)
previous_model.eval()
val_loss = 0.0
val_correct = 0
val_total = 0
val_class_correct = {}
val_class_total = {}
with torch.no_grad():
    for X_val_batch, y_val_batch in val_loader:
        val_outputs = previous_model(X_val_batch).view(-1, output_size-1)

        # # Get raw model predictions (originally only 2 logits)
        # val_outputs = previous_model(X_val_batch)  # Shape: [batch_size, seq_len, 2]
        # # Add a 3rd logit with value 0.0 for all samples
        # third_logit = torch.zeros((*val_outputs.shape[:-1], 1), device=val_outputs.device)  # Shape: [batch_size, seq_len, 1]
        # val_outputs = torch.cat((val_outputs, third_logit), dim=-1)  # Shape: [batch_size, seq_len, 3]
        # # Now apply view operation to match loss function's expected shape
        # val_outputs = val_outputs.view(-1, output_size)  # Correct shape: [batch_size * seq_len, 3]

        val_labels = y_val_batch.view(-1)
        # val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)  # Scale to total loss
        # val_predictions = torch.argmax(val_outputs, dim=-1)
        # val_correct += (val_predictions == val_labels).sum().item()
        # val_total += val_labels.size(0)
        # Compute per-class validation accuracy
        compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
# val_loss /= len(val_loader.dataset)
# val_accuracy = val_correct / val_total
# print(f"val_accuracy = {val_accuracy* 100:.2f}%")

# Compute per-class validation accuracy
# val_classwise_accuracy = {int(c): (val_class_correct[c] / val_class_total[c]) * 100 if val_class_total[c] > 0 else 0 
#                          for c in sorted(val_class_total.keys())}
val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%" 
                            for c in sorted(val_class_total.keys())}

print(f"\nval_classwise_accuracy: {val_classwise_accuracy}")
print(f"val_class_correct: {val_class_correct}")
print(f"val_class_total: {val_class_total}")


del X_train, y_train, X_val, y_val, val_dataset, val_loader, val_classwise_accuracy
torch.cuda.empty_cache()

for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]

Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)


val_classwise_accuracy: {0: '99.54%', 1: '95.88%', 2: '0.00%'}
val_class_correct: {np.int64(0): 238248, np.int64(1): 158370, np.int64(2): 0}
val_class_total: {np.int64(0): 239337, np.int64(1): 165167, np.int64(2): 49496}


In [16]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

In [ ]:
# val_classwise_accuracy: {0: '99.33%', 1: '94.49%', 2: '86.59%'}
# val_class_correct: {np.int64(0): 237743, np.int64(1): 156072, np.int64(2): 42860}
# val_class_total: {np.int64(0): 239337, np.int64(1): 165167, np.int64(2): 49496}
# val_loss: 0.18397733595402754
# val_accuracy: 0.9618392070484582

In [93]:
print(best_model_path)

Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth


In [23]:
import glob, copy

# === Step 1: 定義模型結構與參數 ===
input_size = Number_features  # ✅ 你應該知道你這個 period 的 Number_features
hidden_size = 64
output_size = num_classes   # ✅ num_classes = 3 in Period 2
num_layers = 4
dropout = 0.0

# === Step 2: 準備模型重建 ===
base_model_eval = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout)
new_column_eval = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                             new_output_size=1, num_layers=num_layers, dropout=0.0)
model_template = ProgressiveNeuralNetwork(base_model_eval, new_column_eval)

# === Step 3: 準備驗證資料 ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# === Step 4: 遍歷並補上 val accuracy 與 class-wise accuracy ===
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

for file_path in ckpt_paths:
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=output_size,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# === Step 5: 顯示 summary ===
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_151976\3977337586.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:

✅ Updated: ProgressiveNeuralNetwork_epoch_74.pth | Acc: 0.9617 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try\ProgressiveNeuralNetwork_epoch_74.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_76.pth | Acc: 0.9617 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try\ProgressiveNeuralNetwork_epoch_76.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_83.pth | Acc: 0.9617 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try\ProgressiveNeuralNetwork_epoch_83.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_84.pth | Acc: 0.9618 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try\ProgressiveNeuralNetwork_epoch_84.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_85.pth | Acc: 0.9616 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try\ProgressiveNeuralNetwork_epoch_85.pth
✅ Upd

C:\Users\james\AppData\Local\Temp\ipykernel_151976\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 2 --> Training and saving in __*'2nd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001) ---> Val acc = 96.16 %
### Val-Class-Acc: {0: '99.33%', 1: '93.12%', 2: '82.45%'}

In [147]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/2nd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

In [130]:
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()

### Period 2 --> Training and saving in __*'3rd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.000001) ---> Val acc = 96.05 %
### Val-Class-Acc: {0: '99.39%', 1: '94.34%', 2: '80.74%'}

In [149]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.000001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/3rd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\3rd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

### Period 2 --> Training and saving in __*'4th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005) ---> Val acc = 96.07 %
### Val-Class-Acc: {0: '99.31%', 1: '93.40%', 2: '82.57%'}

In [191]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [194]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/4th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
dic_best_period_2_1st_try = {
    "epoch": 84,
    "train_loss": 0.1628,
    "val_loss": 0.1840,
    "val_accuracy": 0.9618,
    "model_path": r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
}
track_across_runs = [dic_best_period_2_1st_try]
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\4th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa


### Period 2 --> Training and saving in __*'5th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 96.12 %
</pre>
### Val-Class-Acc: {0: '99.38%', 1: '93.37%', 2: '79.93%'}


In [195]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [196]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/5th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
# track_across_runs = [dic_best_period_2_1st_try]
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\5th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

### Period 2 --> Training and saving in __*'6th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 96.03 %
</pre>
### Val-Class-Acc: {0: '99.36%', 1: '94.04%', 2: '80.28%'}

In [197]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [198]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/6th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
# track_across_runs = [dic_best_period_2_1st_try]
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\6th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

### Period 2 --> Training and saving in __*'7th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 96.00 %
</pre>
### Val-Class-Acc: {0: '99.34%', 1: '93.26%', 2: '81.02%'}

In [199]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [200]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/7th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#---------------------
# Initialize the list to store results across runs
# track_across_runs = [dic_best_period_2_1st_try]
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_output_size = 1
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=new_output_size, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\7th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoa

### Period 3 --> Training and saving in __*'1st_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001) ---> Val acc = 95.97 %
### Val-Class-Acc: {0: '89.44%', 1: '96.32%', 2: '90.66%', 3: '97.41%'}

In [170]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )
print(f"\nNumber_features = {Number_features}")
unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")

# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
batch_size= 64 # How many sequences passed at once to the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#-------------------------------------------------------------------------

# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)

# Load the saved state dictionary
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

criterion_val = nn.CrossEntropyLoss()
previous_model.to(device)
previous_model.eval()


# Convert data to tensors # Returns a copy, original is safe
X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val, dtype=torch.long).to(device)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
val_loss = 0.0
val_correct = 0
val_total = 0
val_class_correct = {}
val_class_total = {}
with torch.no_grad():
    for X_val_batch, y_val_batch in val_loader:
        val_outputs = previous_model(X_val_batch).view(-1, output_size)
        val_labels = y_val_batch.view(-1)
        loss_val = criterion_val(val_outputs, val_labels)
        val_loss += loss_val.item() * X_val_batch.size(0)  # Scale to total loss
        val_predictions = torch.argmax(val_outputs, dim=-1)
        val_correct += (val_predictions == val_labels).sum().item()
        val_total += val_labels.size(0)
        # Compute per-class validation accuracy
        compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
val_loss /= len(val_loader.dataset)
val_accuracy = val_correct / val_total
# Compute per-class validation accuracy
val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" 
                            if val_class_total[c] > 0 else "0.00%" 
                            for c in sorted(val_class_total.keys())}
print(f"\nval_classwise_accuracy: {val_classwise_accuracy}")
print(f"val_class_correct: {val_class_correct}")
print(f"val_class_total: {val_class_total}")
print(f"val_loss: {val_loss}")
print(f"val_accuracy: {val_accuracy * 100:.4f}%")

# Convert data to tensors # Returns a copy, original is safe 
X_val = torch.tensor(X_test, dtype=torch.float32).to(device)
y_val = torch.tensor(y_test, dtype=torch.long).to(device)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
val_loss = 0.0
val_correct = 0
val_total = 0
val_class_correct = {}
val_class_total = {}
with torch.no_grad():
    for X_val_batch, y_val_batch in val_loader:
        val_outputs = previous_model(X_val_batch).view(-1, output_size)
        val_labels = y_val_batch.view(-1)
        loss_val = criterion_val(val_outputs, val_labels)
        val_loss += loss_val.item() * X_val_batch.size(0)  # Scale to total loss
        val_predictions = torch.argmax(val_outputs, dim=-1)
        val_correct += (val_predictions == val_labels).sum().item()
        val_total += val_labels.size(0)
        # Compute per-class validation accuracy
        compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
val_loss /= len(val_loader.dataset)
val_accuracy = val_correct / val_total
# Compute per-class validation accuracy
val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" 
                            if val_class_total[c] > 0 else "0.00%" 
                            for c in sorted(val_class_total.keys())}
print(f"\ntest_classwise_accuracy: {val_classwise_accuracy}")
print(f"test_class_correct: {val_class_correct}")
print(f"test_class_total: {val_class_total}")
print(f"test_loss: {val_loss}")
print(f"test_accuracy: {val_accuracy * 100:.4f}%")


del X_train, y_train, X_val, y_val, val_dataset, val_loader, val_classwise_accuracy
del previous_model
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3
Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)


val_classwise_accuracy: {0: '99.33%', 1: '94.49%', 2: '86.59%'}
val_class_correct: {np.int64(0): 237

In [217]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [218]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/1st_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
#---------------------
# Initialize the list to store results across runs
track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try


### Period 3 --> Training and saving in __*'2nd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001) ---> Val acc = 96.19 %
### Val-Class-Acc: {0: '86.09%', 1: '96.30%', 2: '90.74%', 3: '98.15%'}

In [219]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [220]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/2nd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\2nd_try


### Period 3 --> Training and saving in __*'3rd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005) ---> Val acc = 96.26 %
### Val-Class-Acc: {0: '91.17%', 1: '96.34%', 2: '90.74%', 3: '98.08%'} <<<<<<<<<<<<<<<<<<<<<<<<<<<

In [24]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [222]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try


In [25]:
# === PERIOD 3: PNN_CIL ===
# 重建 PNN 架構
input_size = Number_features
hidden_size = 64
output_size = num_classes
num_layers = 4
dropout = 0.0

base_model_eval = ProgressiveNeuralNetwork(
    base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers, dropout),
    new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
)

new_column_eval = PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)

model_template = ProgressiveNeuralNetwork(base_model_eval, new_column_eval)

# 準備驗證資料
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# 檢查點路徑
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# 更新所有 checkpoints（val acc + class acc）
for file_path in ckpt_paths:
    torch.cuda.empty_cache()  # 釋放顯卡記憶體
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=output_size,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# 顯示驗證準確率總覽
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_151976\4281543799.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:

✅ Updated: ProgressiveNeuralNetwork_epoch_1002.pth | Acc: 0.9626 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_738.pth | Acc: 0.9625 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try\ProgressiveNeuralNetwork_epoch_738.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_766.pth | Acc: 0.9624 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try\ProgressiveNeuralNetwork_epoch_766.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_824.pth | Acc: 0.9625 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try\ProgressiveNeuralNetwork_epoch_824.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_841.pth | Acc: 0.9624 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/3rd_try\ProgressiveNeuralNetwork_epoch_8

C:\Users\james\AppData\Local\Temp\ipykernel_151976\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 3 --> Training and saving in __*'4th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 95.67 %
</pre>
### Val-Class-Acc: {0: '86.89%', 1: '96.33%', 2: '90.74%', 3: '97.29%'}

In [223]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [224]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/4th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\4th_try


### Period 3 --> Training and saving in __*'5th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 95.16 %
</pre>
### Val-Class-Acc: {0: '85.25%', 1: '96.38%', 2: '90.61%', 3: '96.79%'}

In [225]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [226]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/5th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\5th_try


### Period 3 --> Training and saving in __*'6th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 92.73 %
</pre>
### Val-Class-Acc: {0: '67.67%', 1: '96.35%', 2: '90.74%', 3: '94.45%'}

In [227]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [228]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/6th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\6th_try


### Period 3 --> Training and saving in __*'7th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.001) ---> Val acc = 95.91 %
### Val-Class-Acc: {0: '83.01%', 1: '96.33%', 2: '90.60%', 3: '97.52%'}

In [229]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [230]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/7th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\7th_try


### Period 3 --> Training and saving in __*'8th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 95.82 %
</pre>
### Val-Class-Acc: {0: '85.11%', 1: '96.29%', 2: '90.74%', 3: '98.02%'}

In [231]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [232]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/8th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_2\1st_try\ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\8th_try


### Period 4 --> Training and saving in __*'1st_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001) ---> Val acc = 92.71 %
### Val-Class-Acc: {0: '89.54%', 1: '93.41%', 2: '86.18%', 3: '95.88%', 4: '87.99%'}

In [26]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [27]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/1st_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth"
#---------------------
# Initialize the list to store results across runs
track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'2nd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001) ---> Val acc = 92.87 %
### Val-Class-Acc: {0: '89.59%', 1: '93.41%', 2: '86.18%', 3: '96.41%', 4: '90.42%'}

In [26]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

In [28]:
# === Post-training Evaluation: Update .pth with val accuracy and class-wise accuracy for PNN Period 4 ===

# ⛳ 重新建立 Period 4 ProgressiveNeuralNetwork 的架構
input_size = Number_features
hidden_size = 64
output_size = num_classes
num_layers = 4
dropout = 0.0

base_model_p1 = BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers, dropout)
column_p2 = PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
pnn_p2 = ProgressiveNeuralNetwork(base_model_p1, column_p2)

column_p3 = PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
pnn_p3 = ProgressiveNeuralNetwork(pnn_p2, column_p3)

column_p4 = PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
model_template = ProgressiveNeuralNetwork(pnn_p3, column_p4)

# 🎯 設定驗證資料與損失函數
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# 📁 模型檔案路徑
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# 🔁 對每個 .pth 模型加上驗證結果
for file_path in ckpt_paths:
    torch.cuda.empty_cache()
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=output_size,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# ✅ 顯示 summary
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_151976\2639005564.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:

✅ Updated: ProgressiveNeuralNetwork_epoch_1836.pth | Acc: 0.9287 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try\ProgressiveNeuralNetwork_epoch_1836.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_1848.pth | Acc: 0.9286 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try\ProgressiveNeuralNetwork_epoch_1848.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_1852.pth | Acc: 0.9287 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try\ProgressiveNeuralNetwork_epoch_1852.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_1918.pth | Acc: 0.9286 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try\ProgressiveNeuralNetwork_epoch_1918.pth
✅ Updated: ProgressiveNeuralNetwork_epoch_1933.pth | Acc: 0.9286 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/2nd_try\ProgressiveNeuralNetwork_

C:\Users\james\AppData\Local\Temp\ipykernel_151976\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 4 --> Training and saving in __*'3rd_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005) ---> Val acc = 92.85 %
### Val-Class-Acc: {0: '89.59%', 1: '93.41%', 2: '86.19%', 3: '95.91%', 4: '90.60%'}

In [30]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [31]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/3rd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'4th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.001) ---> Val acc = 92.71 %
### Val-Class-Acc: {0: '89.39%', 1: '93.41%', 2: '86.17%', 3: '94.75%', 4: '72.25%'}

In [32]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [33]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/4th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'5th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00005, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 92.61 %
</pre>
### Val-Class-Acc: {0: '89.29%', 1: '93.41%', 2: '86.20%', 3: '95.89%', 4: '89.21%'}

In [34]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [35]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.00005 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/5th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'6th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.00001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 92.60 %
</pre>
### Val-Class-Acc: {0: '89.37%', 1: '93.41%', 2: '86.20%', 3: '96.06%', 4: '88.94%'}

In [36]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [37]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.00001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/6th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'7th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.0001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 92.63 %
</pre>
### Val-Class-Acc: {0: '89.31%', 1: '93.41%', 2: '86.20%', 3: '95.88%', 4: '88.86%'}

In [38]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [39]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/7th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

### Period 4 --> Training and saving in __*'8th_try'*__ (ProgressiveNeuralNetwork, num_layers = 4, learning_rate = 0.001, 
<pre>
                                                                                                        use_scheduler=True) ---> Val acc = 92.67 %
</pre>
### Val-Class-Acc: {0: '89.62%', 1: '93.41%', 2: '86.18%', 3: '95.24%', 4: '86.49%'}

In [40]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [41]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.001 # 0.0001
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_4/8th_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
# # best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\1st_try\ProgressiveNeuralNetwork_epoch_99.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, True, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\PNN_CIL\Period_3\3rd_try\ProgressiveNeuralNetwork_epoch_1002.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

##  __Compute FWT & Model Size__

### Period 2

In [ ]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [ ]:
# === 📌 PNN - FWT 計算準備（Period 2） ===
input_size = Number_features
hidden_size = 64
output_size_prev = 2  # Period 1 類別為 [0, 1]
output_size_curr = 3  # Period 2 類別為 [0, 1, 2]
num_layers = 4
dropout = 0.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 📦 載入 Period 1 的 frozen base model ===
prev_model_path = "Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try/BiGRUWithAttention_epoch_1987.pth"
checkpoint = torch.load(prev_model_path, map_location=device)

# === 🧱 建立 previous_model ===
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size_prev, num_layers, dropout).to(device)
previous_model.load_state_dict(checkpoint['model_state_dict'])
previous_model.output_size = output_size_prev  # for view()
del checkpoint
gc.collect()

# === 🧪 建立初始化模型 ===
init_model = BiGRUWithAttention(input_size, hidden_size, output_size_prev, num_layers, dropout).to(device)
init_model.output_size = output_size_prev  # for view()

# === 🧪 準備資料 ===
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# === ✅ 計算 FWT (舊類別：class 0, 1) ===
known_classes = [1]  # PNN 中 0 也是有效類別
fwt, acc_prev, acc_init = compute_fwt_fixed_verbose(previous_model, init_model, X_val_tensor, y_val_tensor, known_classes)
print(f"\n### Period 2:")
print(f"- FWT (PNN Period 2, old classes {known_classes}): {fwt * 100:.2f}%")
print(f"- Accuracy by previous model: {acc_prev * 100:.2f}%")
print(f"- Accuracy by init model:     {acc_init * 100:.2f}%")


/tmp/ipykernel_1897833/3447314965.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(prev_model_path, map_location=device)


📋 Total matching sequences for known classes [1]: 454

### 🔍 FWT Debug Info:
- Total evaluated tokens: 165167
- Correct (PrevModel): 158370 / 165167 → Acc = 0.9588
- Correct (InitModel): 83192 / 165167 → Acc = 0.5037
- FWT = Acc_prev - Acc_init = 0.4552

### Period 2:
- FWT (PNN Period 2, old classes [1]): 45.52%
- Accuracy by previous model: 95.88%
- Accuracy by init model:     50.37%


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00001 # 0.0001
num_epochs= 1 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/TT/PNN_CIL/Period_2/1st_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# Instantiate the previous model
previous_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers).to(device)
#---------------------
best_model_path = r"Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try/BiGRUWithAttention_epoch_1987.pth"

#---------------------
# Initialize the list to store results across runs
track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------

# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)


# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

print_model_info(current_model)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\ncurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try/BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

Total Parameters: 552963
Model Size (float32): 2.11 MB
PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/TT/PNN_CIL/Period_2/1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batc

### Period 3

In [ ]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [ ]:
# === 🧩 PNN Period 3 - FWT 分析 ===
input_size = Number_features
hidden_size = 64
output_size_prev = 3  # Period 2 類別：{0,1,2}
num_layers = 4
dropout = 0.0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 📦 載入 Period 2 的 PNN 模型 ===
prev_model_path = "Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try/ProgressiveNeuralNetwork_epoch_84.pth"
checkpoint = torch.load(prev_model_path, map_location=device)

# === 重建 previous_model (PNN base_model + period2 new_column) ===
previous_model = ProgressiveNeuralNetwork(
    base_model=BiGRUWithAttention(input_size, hidden_size, output_size_prev - 1, num_layers),
    new_column=PNNColumn(input_size, hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
).to(device)
previous_model.load_state_dict(checkpoint['model_state_dict'])
previous_model.output_size = output_size_prev
del checkpoint
gc.collect()

# === 建立 init_model（只包含舊類別）===
init_model = BiGRUWithAttention(input_size, hidden_size, output_size_prev, num_layers, dropout).to(device)
init_model.output_size = output_size_prev

# === FWT 評估資料 ===
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
known_classes = [1, 2]

fwt, acc_prev, acc_init = compute_fwt_fixed_verbose(previous_model, init_model, X_val_tensor, y_val_tensor, known_classes)
print(f"\n### Period 3:")
print(f"- FWT (PNN Period 3, old classes {known_classes}): {fwt * 100:.2f}%")
print(f"- Accuracy by previous model: {acc_prev * 100:.2f}%")
print(f"- Accuracy by init model:     {acc_init * 100:.2f}%")


/tmp/ipykernel_1897833/3902621381.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(prev_model_path, map_location=device)


📋 Total matching sequences for known classes [1, 2]: 454

### 🔍 FWT Debug Info:
- Total evaluated tokens: 228825
- Correct (PrevModel): 219899 / 228825 → Acc = 0.9610
- Correct (InitModel): 84302 / 228825 → Acc = 0.3684
- FWT = Acc_prev - Acc_init = 0.5926

### Period 3:
- FWT (PNN Period 3, old classes [1, 2]): 59.26%
- Accuracy by previous model: 96.10%
- Accuracy by init model:     36.84%


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
lora_r = 4 # Rank of the low-rank update matrices
learning_rate = 0.00005 # 0.0001
num_epochs= 1 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Test_models_size/PNN_CIL/Period_3/3rd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 2, num_layers), # Period 1 model
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
best_model_path = r"Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try/ProgressiveNeuralNetwork_epoch_84.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)
print_model_info(current_model)
train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_2/1st_try/ProgressiveNeuralNetwork_epoch_84.pth

ProgressiveNeuralNetwork(
  (base_model): BiGRUWithAttention(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=2, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
    (attention_fc): Linear(in_features=128, out_features=128, bias=True)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
  )
)

Total Parameters: 837636
Model Size (float32): 3.20 MB
PNN train_and_validate function started.

Existing folder has been removed : Class_Incremental_CL/Classif_Bi

### Period 4

In [ ]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [ ]:
# === 🧩 PNN Period 4 - FWT 分析 ===
input_size = Number_features
hidden_size = 64
output_size_prev = 4  # Period 3 類別：{0,1,2,3}
num_layers = 4
dropout = 0.0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 📦 載入 Period 3 的 PNN 模型 ===
prev_model_path = "Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/1st_try/ProgressiveNeuralNetwork_epoch_1996.pth"
checkpoint = torch.load(prev_model_path, map_location=device)

# === 重建 previous_model (PNN遞迴結構: Period 1→2→3) ===
previous_model = ProgressiveNeuralNetwork(
    base_model=ProgressiveNeuralNetwork(
        base_model=BiGRUWithAttention(input_size, hidden_size, output_size_prev - 2, num_layers),
        new_column=PNNColumn(input_size, hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
    ),
    new_column=PNNColumn(input_size, hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout)
).to(device)
previous_model.load_state_dict(checkpoint['model_state_dict'])
previous_model.output_size = output_size_prev
del checkpoint
gc.collect()

# === 建立 init_model（只包含舊類別）===
init_model = BiGRUWithAttention(input_size, hidden_size, output_size_prev, num_layers, dropout).to(device)
init_model.output_size = output_size_prev

# === FWT 評估資料 ===
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
known_classes = [1, 2, 3]

fwt, acc_prev, acc_init = compute_fwt_fixed_verbose(previous_model, init_model, X_val_tensor, y_val_tensor, known_classes)
print(f"\n### Period 4:")
print(f"- FWT (PNN Period 4, old classes {known_classes}): {fwt * 100:.2f}%")
print(f"- Accuracy by previous model: {acc_prev * 100:.2f}%")
print(f"- Accuracy by init model:     {acc_init * 100:.2f}%")


/tmp/ipykernel_1897833/1951882726.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(prev_model_path, map_location=device)


📋 Total matching sequences for known classes [1, 2, 3]: 454

### 🔍 FWT Debug Info:
- Total evaluated tokens: 385661
- Correct (PrevModel): 361270 / 385661 → Acc = 0.9368
- Correct (InitModel): 90839 / 385661 → Acc = 0.2355
- FWT = Acc_prev - Acc_init = 0.7012

### Period 4:
- FWT (PNN Period 4, old classes [1, 2, 3]): 70.12%
- Accuracy by previous model: 93.68%
- Accuracy by init model:     23.55%


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.00001 # 0.0001
num_epochs= 1 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'ProgressiveNeuralNetwork' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Test_models_size/PNN_CIL/Period_4/2nd_try"))
ensure_folder(model_saving_folder)

#-------------------------------------------------------------------------
# # Instantiate the previous model
previous_model = ProgressiveNeuralNetwork(
                 base_model=ProgressiveNeuralNetwork(
                            base_model=BiGRUWithAttention(input_size, hidden_size, output_size - 3, num_layers), # Period 1 model
                            new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 2
                            ),
                 new_column=PNNColumn(input_size=input_size, hidden_size=hidden_size, new_output_size=1, num_layers=num_layers, dropout=dropout) # New column for period 3
                 ).to(device)
# Load the saved state dictionary
#---------------------
# best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
# best_model_path = best_overall['model_path']
best_model_path = r"Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/1st_try/ProgressiveNeuralNetwork_epoch_1996.pth"
# #---------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#---------------------
#---------------------
previous_model_checkpoint_path = os.path.normpath(best_model_path)
previous_model_checkpoint = torch.load(previous_model_checkpoint_path, map_location=device, weights_only=True)
previous_model.load_state_dict(previous_model_checkpoint['model_state_dict'])
del previous_model_checkpoint
gc.collect()
print(f"Loaded previous model from: \n\t{previous_model_checkpoint_path}")
print(f"\n{previous_model}\n")
#-------------------------------------------------------------------------


# Create the new column for the new class (new_output_size = 1)
new_column = PNNColumn(input_size=input_size, hidden_size=hidden_size, 
                       new_output_size=1, num_layers=num_layers, dropout=0.0)

# Create the progressive model that combines the frozen Period 1 model and the new column
current_model = ProgressiveNeuralNetwork(base_model=previous_model, new_column=new_column)

# Define loss, optimizer, and scheduler.
# Note: Only the new column parameters are trainable, on the new class only.
# weight = torch.zeros(output_size, device=device)
# weight[-1] = 1.0
# # weight = torch.tensor([0.0, 0.0, 1.0], device=device)
# criterion_pnn = nn.CrossEntropyLoss(weight=weight)
criterion_pnn = nn.CrossEntropyLoss()
optimizer_pnn = optim.Adam(new_column.parameters(), lr=learning_rate)
scheduler_pnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_pnn, 'min', factor=0.9, patience=10)

train_and_validate_pnn(current_model, output_size, criterion_pnn, optimizer_pnn, 
                       X_train, y_train, X_val, y_val, scheduler_pnn, False, num_epochs, batch_size, 
                       model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nCurrent_model: \n{current_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]
torch.cuda.empty_cache()


Loaded previous model from: 
	Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/PNN_CIL/Period_3/1st_try/ProgressiveNeuralNetwork_epoch_1996.pth

ProgressiveNeuralNetwork(
  (base_model): ProgressiveNeuralNetwork(
    (base_model): BiGRUWithAttention(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=2, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (new_column): PNNColumn(
      (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
      (attention_fc): Linear(in_features=128, out_features=128, bias=True)
      (fc): Linear(in_features=128, out_features=1, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (lateral_adapter): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (new_column): PNNColumn(
    (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirection

---

## 📊 Comprehensive Test Results:
### (`PNN_CL_Classif_Bi_Dir_GRU_Model.ipynb`)

| Period | Model & Config                                                                 | Validation Accuracy | Class-wise Accuracy                                                  |
|--------|--------------------------------------------------------------------------------|---------------------|-----------------------------------------------------------------------|
| 1      | `BiGRUWithAttention`<br>(num_layers=4)<br>saved in `'1st_try'`               | **98.35%**          | {0: 98.63%, 1: 97.92%}                                               |
| 2      | `ProgressiveNeuralNetwork`<br>(num_layers=4, lr=1e-5)<br>saved in `'1st_try'` | **96.18%**          | {0: 99.33%, 1: 94.49%, 2: 86.59%}                                    |
| 3      | `ProgressiveNeuralNetwork`<br>(num_layers=4, lr=5e-5)<br>saved in `'3rd_try'` | **96.26%**          | {0: 91.17%, 1: 96.34%, 2: 90.74%, 3: 98.08%}                         |
| 4      | `ProgressiveNeuralNetwork`<br>(num_layers=4, lr=1e-5)<br>saved in `'2nd_try'` | **92.87%**          | {0: 89.59%, 1: 93.41%, 2: 86.18%, 3: 96.41%, 4: 90.42%}              |

---


## 📊 Summary: 

### ✔️ Finance - PNN: Validation Summary

| Period | Validation Accuracy | Class-wise Accuracy                                                  |
|--------|---------------------|-----------------------------------------------------------------------|
| 1      | **98.35%**          | {0: 98.63%, 1: 97.92%}                                               |
| 2      | **96.18%**          | {0: 99.33%, 1: 94.49%, 2: 86.59%}                                    |
| 3      | **96.26%**          | {0: 91.17%, 1: 96.34%, 2: 90.74%, 3: 98.08%}                         |
| 4      | **92.87%**          | {0: 89.59%, 1: 93.41%, 2: 86.18%, 3: 96.41%, 4: 90.42%}              |


### 🧠 Continual Learning Metrics

| Period | AA_old (%) | AA_new (%) | BWT (%) | FWT (%) | FWT Classes     | Prev. Model Acc | Init Model Acc |
|--------|------------|------------|---------|---------|------------------|------------------|-----------------|
| 2      | 96.91%     | 86.59%     | -1.37%  | 45.52%  | [1]              | 95.88%           | 50.37%          |
| 3      | 92.75%     | 98.08%     | -0.72%  | 59.26%  | [1, 2]           | 96.10%           | 36.84%          |
| 4      | 91.90%     | 90.42%     | -2.69%  | 70.12%  | [1, 2, 3]        | 93.68%           | 23.55%          |


### 📦 Model Size per Period

| Period | Output Size | Total Params | Δ Params vs Prev | Δ % vs Prev | Model Size (float32) |
|--------|-------------|--------------|------------------|-------------|-----------------------|
| 1      | 2           | 268,290      | —                | —           | 1.02 MB               |
| 2      | 3           | 552,963      | +284,673         | +106.08%    | 2.11 MB               |
| 3      | 4           | 837,636      | +284,673         | +51.49%     | 3.20 MB               |
| 4      | 5           | 1,122,309    | +284,673         | +34.00%     | 4.28 MB               |

**📈 Model Growth Rate (MGR) = (1,122,309 - 268,290) / (268,290 × 3) ≈ +106.03%**

**📈 Max trainable ratio ≈ 51.48%**